<a href="https://colab.research.google.com/github/mamecoumba-konteye/self-healing-telecom-ia/blob/main/AWID3_Pipeline_Complet_Metriques_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time
import numpy as np
import pandas as pd

base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
os.makedirs(checkpoint_dir_v4, exist_ok=True)
window_size = 20

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}

known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}

non_generalisable_keywords = [
    'seq', 'ack', 'checksum', 'payload', 'kck', 'kek', 'gtk', 'igtk', 'file_data',
    'analysis.rto', 'frame.time', 'frame.number', 'radiotap.', 'wlan_radio.',
]
identity_keywords = ['bssid', 'wlan.ta', 'wlan.sa', 'wlan.ra', 'wlan.da', 'ip.src', 'ip.dst']

raw_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        df_raw = read_csv_safe(os.path.join(folder_path, f))
        df_raw = df_raw.reset_index(drop=True)

        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw
            gc.collect()
            continue

        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        df_slice = df_raw.iloc[start:end].reset_index(drop=True)
        del df_raw
        gc.collect()

        cols_to_drop = [c for c in df_slice.columns if any(k in c for k in non_generalisable_keywords)]
        df_slice = df_slice.drop(columns=cols_to_drop)
        identity_found = [c for c in df_slice.columns if any(k in c for k in identity_keywords)]
        df_slice = df_slice.drop(columns=identity_found)

        file_id = f'{attack_name}__{f}'
        df_slice['source_attack'] = attack_name
        df_slice['source_file'] = file_id
        raw_slices[file_id] = df_slice
        gc.collect()

    print(f'{attack_name}: {len(files)} fichier(s) collecte(s) avec contexte complet')

df_combined = pd.concat(raw_slices.values(), ignore_index=True)
del raw_slices
gc.collect()

print("\nDimensions combinees :", df_combined.shape)
print("Repartition par attaque :")
print(df_combined['source_attack'].value_counts())

object_cols = df_combined.select_dtypes(include='object').columns.tolist()
object_cols = [c for c in object_cols if c not in ['source_attack', 'source_file', 'Label']]
print(f"\nColonnes texte a uniformiser avant sauvegarde : {len(object_cols)}")
for col in object_cols:
    df_combined[col] = df_combined[col].astype('string')

df_combined.to_parquet(f'{checkpoint_dir_v4}/df_raw_v4.parquet')
print("\nCheckpoint intermediaire sauvegarde (avant nettoyage).")

Mounted at /content/drive


/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,115,116,117,118,153,155,157,158,159,160,161,162,163,164,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,247,248,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,86,90,91,92,93,94,96,98,99,100,101,102,103,104,105,107,108,109,110,111,112,113,114,115,116,117,153,155,157,158,159,160,161,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,250,251) have mixed types. Specify dtype option on import or set low_

Deauth: 5 fichier(s) collecte(s) avec contexte complet


/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,90,91,92,98,99,100,101,102,103,104,105,107,108,109,110,111,115,119,120,135,137,140,144,146,148,149,151,167,168,178,180,183,184,185,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (71,73,77,78,79,80,81,82,83,84,85,86,90,91,92,93,94,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,122,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,191,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarn

Disas: 5 fichier(s) collecte(s) avec contexte complet


/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,116,117,118,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,118,132,166,167,168,183,184,185,189,190,192,193,194,195,196,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (20,21,43,44,45,62,65,

... (17 outputs left out)

/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,87,90,91,92,93,98,99,100,101,102,103,104,105,107,108,109,110,111,115,118,166,167,168,172,178,180,181,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,242,243,244,247,248,249,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,90,91,92,93,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,116,117,118,166,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,249) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (71,73,77,78,79,80,86,93,112,113,114,116,117,118,166,167,168,183,184,185,189,190,

Evil_Twin: 5 fichier(s) collecte(s) avec contexte complet


/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (23,24,48,50,51,81,82,83,84,85,86,90,91,92,93,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (82,84,85,86,87,90,91,92,93,94,96,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,249,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/4222876034.py:16: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,87,90,91,92,93,97,98,99,100,101,102,103,

Website_spoofing: 5 fichier(s) collecte(s) avec contexte complet

Dimensions combinees : (3092490, 202)
Repartition par attaque :
source_attack
SSDP                250000
Website_spoofing    249966
Disas               249732
Deauth              249536
Re-Assoc            249251
Kr00k               248669
Botnet              248450
Rogue_AP            248091
Evil_Twin           246190
Malware             245468
SSH                 241407
SQL_Injection       202947
Krack               162783
Name: count, dtype: int64

Colonnes texte a uniformiser avant sauvegarde : 134

Checkpoint intermediaire sauvegarde (avant nettoyage).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
protected_cols = ['Label', 'source_attack', 'source_file']

df_raw = pd.read_parquet(f'{checkpoint_dir_v4}/df_raw_v4.parquet')
print("Dimensions chargees :", df_raw.shape)

missing_pct = df_raw.isnull().mean() * 100
to_drop_missing = set(missing_pct[missing_pct > 90].index) - set(protected_cols)
zero_variance = [c for c in df_raw.columns if c not in protected_cols
                  and df_raw[c].nunique(dropna=True) <= 1]
df_clean = df_raw.drop(columns=[c for c in (to_drop_missing | set(zero_variance)) if c in df_raw.columns])
del df_raw
gc.collect()

print("Colonnes retirees (plus de 90 pourcent vides ou sans variance) :", len(to_drop_missing | set(zero_variance)))
print("Dimensions apres ce premier filtre :", df_clean.shape)

def clean_tcp_flag(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if any(c.isalpha() for c in s): return np.nan
    parts = s.split('-')
    try:
        nums = [int(float(p)) for p in parts if p != '']
        return max(nums) if nums else np.nan
    except ValueError:
        return np.nan

def clean_numeric_dashed(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if any(c.isalpha() for c in s): return np.nan
    parts = s.split('-')
    try:
        nums = [float(p) for p in parts if p != '']
        return max(nums) if nums else np.nan
    except ValueError:
        return np.nan

def clean_repeated_category(val, known_values):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    for kv in sorted(known_values, key=len, reverse=True):
        if s == kv or (s.endswith(kv) and s[:-len(kv)].endswith('-')):
            return kv
    return s

flag_cols = ['tcp.flags.syn', 'tcp.flags.fin', 'tcp.flags.push', 'tcp.flags.reset']
for col in flag_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_tcp_flag)

numeric_dashed_cols = ['ip.proto', 'ip.ttl', 'ip.version', 'tcp.option_len',
                        'tcp.srcport', 'tcp.dstport', 'tcp.time_delta', 'tcp.time_relative',
                        'udp.srcport', 'udp.dstport', 'udp.length', 'udp.time_relative', 'udp.time_delta']
for col in numeric_dashed_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_numeric_dashed)

known_methods = ['CONNECT', 'GET', 'HEAD', 'M-SEARCH', 'NOTIFY', 'OPTIONS', 'POST', 'PUT', 'SUBSCRIBE', 'UNSUBSCRIBE']
known_versions = ['HTTP/1.1', 'HTTP/1.0', 'HTTP/2.0', 'HTTP/2']
if 'http.request.method' in df_clean.columns:
    df_clean['http.request.method'] = df_clean['http.request.method'].apply(lambda v: clean_repeated_category(v, known_methods))
if 'http.request.version' in df_clean.columns:
    df_clean['http.request.version'] = df_clean['http.request.version'].apply(lambda v: clean_repeated_category(v, known_versions))

categorical_cols = [c for c in ['wlan.fc.ds', 'llc', 'tcp.analysis', 'http.request.method', 'http.request.version']
                     if c in df_clean.columns]
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, dummy_na=True)
del df_clean
gc.collect()

feature_cols_temp = [c for c in df_encoded.columns if c not in protected_cols]
df_encoded[feature_cols_temp] = df_encoded[feature_cols_temp].fillna(0)

object_cols_remaining = [c for c in df_encoded.columns if c not in protected_cols and df_encoded[c].dtype in ('object', 'string')]
for col in object_cols_remaining:
    df_encoded[col] = df_encoded[col].apply(clean_numeric_dashed).fillna(0)
    print(f"Colonne texte residuelle nettoyee numeriquement : {col}")

import re
suspicious_patterns = re.compile(r'(_\d{1,2}-\w{3}$)|(_\d{2}-\d{2}-\d{2}$)|(_\d+\.0$)|(_\d+$)')
encoded_from = {c: c.split('_')[0] for c in df_encoded.columns if '_' in c and c not in protected_cols}
base_groups = {}
for col, base in encoded_from.items():
    base_groups.setdefault(base, []).append(col)

print("\n=== Verification : bases avec plusieurs variantes numeriques suspectes ===")
for base, cols in base_groups.items():
    suspects = [c for c in cols if suspicious_patterns.search(c)]
    if len(suspects) >= 2:
        print(f"  {base} : {suspects}")

corrupted_variants = ['tcp.analysis_1.0', 'tcp.analysis_01-01-01', 'tcp.analysis_01-Jan', 'tcp.analysis_1']
present = [c for c in corrupted_variants if c in df_encoded.columns]
if present:
    df_encoded['tcp.analysis_clean'] = df_encoded[present].max(axis=1).astype(int)
    df_encoded = df_encoded.drop(columns=present)
if 'tcp.analysis_<NA>' in df_encoded.columns:
    df_encoded = df_encoded.drop(columns=['tcp.analysis_<NA>'])

feature_cols = [c for c in df_encoded.columns if c not in protected_cols]
zero_var_final = [c for c in feature_cols if df_encoded[c].nunique() <= 1]
df_encoded = df_encoded.drop(columns=zero_var_final)
feature_cols = [c for c in feature_cols if c not in zero_var_final]

print("\nDimensions finales :", df_encoded.shape, "| Features :", len(feature_cols))
print("Colonnes constantes retirees :", zero_var_final)

df_encoded.to_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
print("\nCheckpoint sauvegarde :", f'{checkpoint_dir_v4}/df_encoded_v4.parquet')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dimensions chargees : (3092490, 202)
Colonnes retirees (plus de 90 pourcent vides ou sans variance) : 174
Dimensions apres ce premier filtre : (3092490, 28)

=== Verification : bases avec plusieurs variantes numeriques suspectes ===
  tcp.analysis : ['tcp.analysis_01-01-01', 'tcp.analysis_01-Jan', 'tcp.analysis_1', 'tcp.analysis_1.0']

Dimensions finales : (3092490, 34) | Features : 31
Colonnes constantes retirees : ['wlan.fc.ds_<NA>']

Checkpoint sauvegarde : /content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4/df_encoded_v4.parquet


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
protected_cols = ['Label', 'source_attack', 'source_file']
feature_cols = [c for c in df_encoded.columns if c not in protected_cols]
print("Nombre de features :", len(feature_cols))

X_windows_max, y_windows = [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_windows_max.append(features[s:s + window_size].max(axis=0))
        wl = labels[s:s + window_size]
        y_windows.append(attack_name if (wl != 'Normal').any() else 'Normal')
del df_encoded
gc.collect()

le_max = LabelEncoder()
y_max_encoded = le_max.fit_transform(y_windows)
X_max = np.array(X_windows_max, dtype=np.float32)
del X_windows_max
gc.collect()

X_train, X_test, y_train, y_test = train_test_split(
    X_max, y_max_encoded, test_size=0.2, random_state=42, stratify=y_max_encoded
)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

xgb_combo = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, eval_metric='mlogloss', random_state=42)
xgb_combo.fit(X_train_res, y_train_res)
y_pred = xgb_combo.predict(X_test)
y_proba = xgb_combo.predict_proba(X_test)

y_test_binarized = label_binarize(y_test, classes=np.arange(len(le_max.classes_)))
roc_auc_macro = roc_auc_score(y_test_binarized, y_proba, average='macro', multi_class='ovr')
roc_auc_par_classe = roc_auc_score(y_test_binarized, y_proba, average=None, multi_class='ovr')

print("\n=== Metriques globales (XGBoost, max+SMOTE) ===")
print(f"Accuracy         : {accuracy_score(y_test, y_pred):.4f}")
print(f"Rappel (macro)   : {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-score (macro) : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"ROC-AUC (macro)  : {roc_auc_macro:.4f}")

print("\n=== ROC-AUC par classe ===")
for cls, auc in zip(le_max.classes_, roc_auc_par_classe):
    print(f"  {cls:20s} : {auc:.4f}")

print("\n=== Rapport de classification complet (par classe) ===")
print(classification_report(y_test, y_pred, target_names=le_max.classes_))

print("\n=== Matrice de confusion ===")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=le_max.classes_, columns=le_max.classes_)
print(cm_df)

joblib.dump(xgb_combo, f'{checkpoint_dir_v4}/xgb_combo_v4.joblib')
with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'wb') as f:
    pickle.dump({'le': le_max, 'feature_cols': feature_cols}, f)
cm_df.to_csv(f'{checkpoint_dir_v4}/confusion_matrix_xgb_combo_v4.csv')
pd.Series(dict(zip(le_max.classes_, roc_auc_par_classe))).to_csv(f'{checkpoint_dir_v4}/roc_auc_xgb_combo_v4.csv')
print("\nModele, encodeur, matrice de confusion et ROC-AUC sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Nombre de features : 31

=== Metriques globales (XGBoost, max+SMOTE) ===
Accuracy         : 0.8729
Rappel (macro)   : 0.7951
F1-score (macro) : 0.7607
ROC-AUC (macro)  : 0.9897

=== ROC-AUC par classe ===
  Botnet               : 0.9966
  Deauth               : 0.9881
  Disas                : 0.9779
  Evil_Twin            : 0.9954
  Kr00k                : 0.9814
  Krack                : 0.9979
  Malware              : 0.9998
  Normal               : 0.9782
  Re-Assoc             : 0.9904
  Rogue_AP             : 0.9755
  SQL_Injection        : 0.9838
  SSDP                 : 1.0000
  SSH                  : 0.9910
  Website_spoofing     : 0.9994

=== Rapport de classification complet (par classe) ===
                  precision    recall  f1-score   support

          Botnet       0.86      0.87      0.86      1058
          Deauth       0.69      0.84      0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras import layers, models

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
protected_cols = ['Label', 'source_attack', 'source_file']
feature_cols = [c for c in df_encoded.columns if c not in protected_cols]

X_windows_seq, y_windows = [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_windows_seq.append(features[s:s + window_size])
        wl = labels[s:s + window_size]
        y_windows.append(attack_name if (wl != 'Normal').any() else 'Normal')
del df_encoded
gc.collect()

le = LabelEncoder()
y_encoded = le.fit_transform(y_windows)
X_seq = np.array(X_windows_seq, dtype=np.float32)
del X_windows_seq
gc.collect()

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
del X_seq
gc.collect()

n_timesteps, n_features = X_train.shape[1], X_train.shape[2]
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, n_features))
X_train_scaled = scaler.transform(X_train.reshape(-1, n_features)).astype(np.float32).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).astype(np.float32).reshape(X_test.shape)

class_weights_raw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tempered = {i: w ** 0.5 for i, w in enumerate(class_weights_raw)}

lstm_model = models.Sequential([
    layers.LSTM(64, activation='tanh', input_shape=(n_timesteps, n_features)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(le.classes_), activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

lstm_model.fit(
    X_train_scaled, y_train,
    validation_split=0.1, epochs=15, batch_size=64,
    class_weight=class_weights_tempered, verbose=1
)

y_pred_proba_lstm = lstm_model.predict(X_test_scaled, batch_size=256)
y_pred_lstm = y_pred_proba_lstm.argmax(axis=1)

y_test_binarized = label_binarize(y_test, classes=np.arange(len(le.classes_)))
roc_auc_macro = roc_auc_score(y_test_binarized, y_pred_proba_lstm, average='macro', multi_class='ovr')
roc_auc_par_classe = roc_auc_score(y_test_binarized, y_pred_proba_lstm, average=None, multi_class='ovr')

print("\n=== Metriques globales (LSTM) ===")
print(f"Accuracy         : {accuracy_score(y_test, y_pred_lstm):.4f}")
print(f"Rappel (macro)   : {recall_score(y_test, y_pred_lstm, average='macro'):.4f}")
print(f"F1-score (macro) : {f1_score(y_test, y_pred_lstm, average='macro'):.4f}")
print(f"ROC-AUC (macro)  : {roc_auc_macro:.4f}")

print("\n=== ROC-AUC par classe ===")
for cls, auc in zip(le.classes_, roc_auc_par_classe):
    print(f"  {cls:20s} : {auc:.4f}")

print("\n=== Rapport de classification complet (par classe) ===")
print(classification_report(y_test, y_pred_lstm, target_names=le.classes_))

print("\n=== Matrice de confusion ===")
cm_lstm = confusion_matrix(y_test, y_pred_lstm)
cm_lstm_df = pd.DataFrame(cm_lstm, index=le.classes_, columns=le.classes_)
print(cm_lstm_df)

lstm_model.save(f'{checkpoint_dir_v4}/lstm_v4.keras')
with open(f'{checkpoint_dir_v4}/scaler_lstm_v4.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'le': le, 'feature_cols': feature_cols}, f)
cm_lstm_df.to_csv(f'{checkpoint_dir_v4}/confusion_matrix_lstm_v4.csv')
pd.Series(dict(zip(le.classes_, roc_auc_par_classe))).to_csv(f'{checkpoint_dir_v4}/roc_auc_lstm_v4.csv')
print("\nModele, scaler, matrice de confusion et ROC-AUC sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.7319 - loss: 0.7632 - val_accuracy: 0.8191 - val_loss: 0.5590
Epoch 2/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8138 - loss: 0.4792 - val_accuracy: 0.8275 - val_loss: 0.4667
Epoch 3/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8397 - loss: 0.3974 - val_accuracy: 0.8377 - val_loss: 0.4451
Epoch 4/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8561 - loss: 0.3483 - val_accuracy: 0.8744 - val_loss: 0.3453
Epoch 5/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8681 - loss: 0.3149 - val_accuracy: 0.8707 - val_loss: 0.3356
Epoch 6/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8778 - loss: 0.2906 - val_accuracy: 0.8589 - val_loss: 0.3712
Epoch 7/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8832 - loss: 0.2717 - val_accuracy: 0.8846 - val_loss: 0.3012
Epoch 8/15
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8864 - loss: 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch_geometric

import os, gc, time, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}
mac_cols = ['wlan.sa', 'wlan.da', 'wlan.ta', 'wlan.ra', 'wlan.bssid']
role_cols_map = {'wlan.sa': 0, 'wlan.da': 1, 'wlan.ta': 2, 'wlan.ra': 3, 'wlan.bssid': 4}

mac_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        df_raw = read_csv_safe(os.path.join(folder_path, f)).reset_index(drop=True)
        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw; gc.collect(); continue
        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        cols_present = [c for c in mac_cols if c in df_raw.columns] + ['Label']
        df_mac_slice = df_raw.iloc[start:end][cols_present].reset_index(drop=True)
        file_id = f'{attack_name}__{f}'
        df_mac_slice['source_file'] = file_id
        mac_slices[file_id] = df_mac_slice
        del df_raw; gc.collect()
    print(f'{attack_name}: adresses MAC extraites')

df_mac_combined = pd.concat(mac_slices.values(), ignore_index=True)
del mac_slices
gc.collect()

le_gnn = LabelEncoder()

graph_data_list = []
for file_id, group in df_mac_combined.groupby('source_file', sort=False):
    mac_cols_present = [c for c in mac_cols if c in group.columns]
    attack_label = file_id.split('__')[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window_df = group.iloc[s:s + window_size]
        edges = set()
        role_counts = {}
        for _, row in window_df.iterrows():
            present = [(c, row[c]) for c in mac_cols_present if pd.notna(row[c])]
            addrs = [a for _, a in present]
            for col, mac in present:
                if mac not in role_counts:
                    role_counts[mac] = [0, 0, 0, 0, 0]
                role_counts[mac][role_cols_map[col]] += 1
            if len(addrs) >= 2:
                edges.add(tuple(sorted(addrs[:2])))
        node_list = sorted(role_counts.keys()) if role_counts else ['isole']
        if 'isole' in node_list:
            role_counts['isole'] = [0, 0, 0, 0, 0]
        node_idx = {mac: idx for idx, mac in enumerate(node_list)}
        edge_index_list = []
        for a, b in edges:
            edge_index_list.append([node_idx[a], node_idx[b]])
            edge_index_list.append([node_idx[b], node_idx[a]])
        x = torch.tensor([role_counts[mac] for mac in node_list], dtype=torch.float)
        edge_index = (torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
                      if edge_index_list else torch.zeros((2, 0), dtype=torch.long))
        window_labels = window_df['Label'].values
        label_name = attack_label if (window_labels != 'Normal').any() else 'Normal'
        graph_data_list.append((label_name, Data(x=x, edge_index=edge_index)))

del df_mac_combined
gc.collect()

all_labels = [lbl for lbl, _ in graph_data_list]
y_gnn_encoded = le_gnn.fit_transform(all_labels)
for (_, data), y_val in zip(graph_data_list, y_gnn_encoded):
    data.y = torch.tensor([y_val])
graph_data_list = [data for _, data in graph_data_list]

print('\nGraphes construits :', len(graph_data_list))

graph_indices = np.arange(len(graph_data_list))
train_idx_g, test_idx_g = train_test_split(graph_indices, test_size=0.2, random_state=42, stratify=y_gnn_encoded)
train_graphs = [graph_data_list[i] for i in train_idx_g]
test_graphs = [graph_data_list[i] for i in test_idx_g]
print('Train :', len(train_graphs), ' Test :', len(test_graphs))

class GCN(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gnn_model = GCN(num_features=5, num_classes=len(le_gnn.classes_)).to(device)
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.01)

train_loader = PyGDataLoader(train_graphs, batch_size=128, shuffle=True)
test_loader = PyGDataLoader(test_graphs, batch_size=256, shuffle=False)

for epoch in range(15):
    gnn_model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = gnn_model(batch.x, batch.edge_index, batch.batch)
        loss = F.cross_entropy(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    gnn_model.eval()
    correct = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pred = gnn_model(batch.x, batch.edge_index, batch.batch).argmax(dim=1)
            correct += (pred == batch.y).sum().item()
    print(f"Epoch {epoch+1}/15 - loss: {total_loss/len(train_loader):.4f} - val_accuracy: {correct/len(test_graphs):.4f}")

gnn_model.eval()
y_pred_gnn, y_true_gnn, y_proba_gnn = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = gnn_model(batch.x, batch.edge_index, batch.batch)
        proba = F.softmax(out, dim=1)
        pred = out.argmax(dim=1)
        y_pred_gnn.extend(pred.cpu().numpy())
        y_true_gnn.extend(batch.y.cpu().numpy())
        y_proba_gnn.extend(proba.cpu().numpy())
y_pred_gnn, y_true_gnn, y_proba_gnn = np.array(y_pred_gnn), np.array(y_true_gnn), np.array(y_proba_gnn)

y_test_binarized = label_binarize(y_true_gnn, classes=np.arange(len(le_gnn.classes_)))
roc_auc_macro = roc_auc_score(y_test_binarized, y_proba_gnn, average='macro', multi_class='ovr')
roc_auc_par_classe = roc_auc_score(y_test_binarized, y_proba_gnn, average=None, multi_class='ovr')

print("\n=== Metriques globales (GNN) ===")
print(f"Accuracy         : {accuracy_score(y_true_gnn, y_pred_gnn):.4f}")
print(f"Rappel (macro)   : {recall_score(y_true_gnn, y_pred_gnn, average='macro'):.4f}")
print(f"F1-score (macro) : {f1_score(y_true_gnn, y_pred_gnn, average='macro'):.4f}")
print(f"ROC-AUC (macro)  : {roc_auc_macro:.4f}")

print("\n=== ROC-AUC par classe ===")
for cls, auc in zip(le_gnn.classes_, roc_auc_par_classe):
    print(f"  {cls:20s} : {auc:.4f}")

print("\n=== Rapport de classification complet (par classe) ===")
print(classification_report(y_true_gnn, y_pred_gnn, target_names=le_gnn.classes_))

print("\n=== Matrice de confusion ===")
cm_gnn = confusion_matrix(y_true_gnn, y_pred_gnn)
cm_gnn_df = pd.DataFrame(cm_gnn, index=le_gnn.classes_, columns=le_gnn.classes_)
print(cm_gnn_df)

torch.save(gnn_model.state_dict(), f'{checkpoint_dir_v4}/gnn_v4.pt')
with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'wb') as f:
    pickle.dump({'le': le_gnn}, f)
cm_gnn_df.to_csv(f'{checkpoint_dir_v4}/confusion_matrix_gnn_v4.csv')
pd.Series(dict(zip(le_gnn.classes_, roc_auc_par_classe))).to_csv(f'{checkpoint_dir_v4}/roc_auc_gnn_v4.csv')
print("\nModele, encodeur, matrice de confusion et ROC-AUC sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.7 MB/s eta 0:00:00


/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,115,116,117,118,153,155,157,158,159,160,161,162,163,164,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,247,248,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,86,90,91,92,93,94,96,98,99,100,101,102,103,104,105,107,108,109,110,111,112,113,114,115,116,117,153,155,157,158,159,160,161,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,250,251) have mixed types. Specify dtype option on import or set low_

Deauth: adresses MAC extraites


/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,90,91,92,98,99,100,101,102,103,104,105,107,108,109,110,111,115,119,120,135,137,140,144,146,148,149,151,167,168,178,180,183,184,185,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (71,73,77,78,79,80,81,82,83,84,85,86,90,91,92,93,94,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,122,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,191,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/1317643347.py:25: DtypeWarn

Disas: adresses MAC extraites


/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,116,117,118,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,118,132,166,167,168,183,184,185,189,190,192,193,194,195,196,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (20,21,43,44,45,62,65,

... (18 outputs left out)

Evil_Twin: adresses MAC extraites


/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (23,24,48,50,51,81,82,83,84,85,86,90,91,92,93,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (82,84,85,86,87,90,91,92,93,94,96,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,249,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/1317643347.py:25: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,87,90,91,92,93,97,98,99,100,101,102,103,

Website_spoofing: adresses MAC extraites

Graphes construits : 154606
Train : 123684  Test : 30922
Epoch 1/15 - loss: 0.9777 - val_accuracy: 0.7226
Epoch 2/15 - loss: 0.7821 - val_accuracy: 0.7450
Epoch 3/15 - loss: 0.7390 - val_accuracy: 0.7373
Epoch 4/15 - loss: 0.7139 - val_accuracy: 0.7481
Epoch 5/15 - loss: 0.7022 - val_accuracy: 0.7596
Epoch 6/15 - loss: 0.6891 - val_accuracy: 0.7454
Epoch 7/15 - loss: 0.6844 - val_accuracy: 0.7625
Epoch 8/15 - loss: 0.6796 - val_accuracy: 0.7577
Epoch 9/15 - loss: 0.6746 - val_accuracy: 0.7630
Epoch 10/15 - loss: 0.6723 - val_accuracy: 0.7637
Epoch 11/15 - loss: 0.6680 - val_accuracy: 0.7572
Epoch 12/15 - loss: 0.6669 - val_accuracy: 0.7666
Epoch 13/15 - loss: 0.6631 - val_accuracy: 0.7597
Epoch 14/15 - loss: 0.6597 - val_accuracy: 0.7605
Epoch 15/15 - loss: 0.6575 - val_accuracy: 0.7600

=== Metriques globales (GNN) ===
Accuracy         : 0.7600
Rappel (macro)   : 0.5230
F1-score (macro) : 0.5227
ROC-AUC (macro)  : 0.9609

=== ROC-AUC par class

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
protected_cols = ['Label', 'source_attack', 'source_file']
feature_cols = [c for c in df_encoded.columns if c not in protected_cols]

xgb_combo = joblib.load(f'{checkpoint_dir_v4}/xgb_combo_v4.joblib')
with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    le_max = pickle.load(f)['le']

from tensorflow.keras import models as keras_models
lstm_model = keras_models.load_model(f'{checkpoint_dir_v4}/lstm_v4.keras')
with open(f'{checkpoint_dir_v4}/scaler_lstm_v4.pkl', 'rb') as f:
    saved_lstm = pickle.load(f)
scaler = saved_lstm['scaler']
le = saved_lstm['le']

with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'rb') as f:
    le_gnn = pickle.load(f)['le']

print("Classes identiques entre les 3 encodeurs :",
      list(le.classes_) == list(le_max.classes_) == list(le_gnn.classes_))

X_max_full, X_seq_full, y_windows_full = [], [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window = features[s:s + window_size]
        X_max_full.append(window.max(axis=0))
        X_seq_full.append(window)
        wl = labels[s:s + window_size]
        y_windows_full.append(attack_name if (wl != 'Normal').any() else 'Normal')
del df_encoded
gc.collect()

y_full_encoded = le.transform(y_windows_full)
all_idx = np.arange(len(y_full_encoded))
_, test_idx = train_test_split(all_idx, test_size=0.2, random_state=42, stratify=y_full_encoded)

X_max_test = np.array([X_max_full[i] for i in test_idx], dtype=np.float32)
X_seq_test = np.array([X_seq_full[i] for i in test_idx], dtype=np.float32)
y_test_aligned = y_full_encoded[test_idx]
del X_max_full, X_seq_full
gc.collect()

xgb_proba = xgb_combo.predict_proba(X_max_test)

n_timesteps, n_feat = X_seq_test.shape[1], X_seq_test.shape[2]
X_seq_test_scaled = scaler.transform(X_seq_test.reshape(-1, n_feat)).astype(np.float32).reshape(X_seq_test.shape)
lstm_proba = lstm_model.predict(X_seq_test_scaled, batch_size=256)

print("\nVerification dimensions : xgb_proba", xgb_proba.shape, "| lstm_proba", lstm_proba.shape)

np.savez(f'{checkpoint_dir_v4}/xgb_lstm_proba_v4.npz',
         xgb_proba=xgb_proba, lstm_proba=lstm_proba, y_test=y_test_aligned, test_idx=test_idx)
print("Probabilites XGBoost et LSTM sauvegardees (etape intermediaire).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Classes identiques entre les 3 encodeurs : True
121/121 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Verification dimensions : xgb_proba (30922, 14) | lstm_proba (30922, 14)
Probabilites XGBoost et LSTM sauvegardees (etape intermediaire).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch_geometric

import os, gc, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.metrics import (accuracy_score, recall_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.preprocessing import label_binarize

base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}
mac_cols = ['wlan.sa', 'wlan.da', 'wlan.ta', 'wlan.ra', 'wlan.bssid']
role_cols_map = {'wlan.sa': 0, 'wlan.da': 1, 'wlan.ta': 2, 'wlan.ra': 3, 'wlan.bssid': 4}

mac_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        df_raw = read_csv_safe(os.path.join(folder_path, f)).reset_index(drop=True)
        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw; gc.collect(); continue
        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        cols_present = [c for c in mac_cols if c in df_raw.columns] + ['Label']
        df_mac_slice = df_raw.iloc[start:end][cols_present].reset_index(drop=True)
        file_id = f'{attack_name}__{f}'
        df_mac_slice['source_file'] = file_id
        mac_slices[file_id] = df_mac_slice
        del df_raw; gc.collect()
    print(f'{attack_name}: adresses MAC extraites')

df_mac_combined = pd.concat(mac_slices.values(), ignore_index=True)
del mac_slices
gc.collect()

with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'rb') as f:
    le_gnn = pickle.load(f)['le']

graph_data_list = []
for file_id, group in df_mac_combined.groupby('source_file', sort=False):
    mac_cols_present = [c for c in mac_cols if c in group.columns]
    attack_label = file_id.split('__')[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window_df = group.iloc[s:s + window_size]
        edges = set()
        role_counts = {}
        for _, row in window_df.iterrows():
            present = [(c, row[c]) for c in mac_cols_present if pd.notna(row[c])]
            addrs = [a for _, a in present]
            for col, mac in present:
                if mac not in role_counts:
                    role_counts[mac] = [0, 0, 0, 0, 0]
                role_counts[mac][role_cols_map[col]] += 1
            if len(addrs) >= 2:
                edges.add(tuple(sorted(addrs[:2])))
        node_list = sorted(role_counts.keys()) if role_counts else ['isole']
        if 'isole' in node_list:
            role_counts['isole'] = [0, 0, 0, 0, 0]
        node_idx = {mac: idx for idx, mac in enumerate(node_list)}
        edge_index_list = []
        for a, b in edges:
            edge_index_list.append([node_idx[a], node_idx[b]])
            edge_index_list.append([node_idx[b], node_idx[a]])
        x = torch.tensor([role_counts[mac] for mac in node_list], dtype=torch.float)
        edge_index = (torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
                      if edge_index_list else torch.zeros((2, 0), dtype=torch.long))
        window_labels = window_df['Label'].values
        label_name = attack_label if (window_labels != 'Normal').any() else 'Normal'
        graph_data_list.append(Data(x=x, edge_index=edge_index, y=torch.tensor([le_gnn.transform([label_name])[0]])))

del df_mac_combined
gc.collect()

print("Graphes reconstruits :", len(graph_data_list))

data_xl = np.load(f'{checkpoint_dir_v4}/xgb_lstm_proba_v4.npz')
test_idx = data_xl['test_idx']
y_test_aligned = data_xl['y_test']
xgb_proba = data_xl['xgb_proba']
lstm_proba = data_xl['lstm_proba']

test_graphs_aligned = [graph_data_list[i] for i in test_idx]
y_gnn_test = np.array([g.y.item() for g in test_graphs_aligned])
print("Verification alignement GNN et hybride (doit etre True) :", (y_gnn_test == y_test_aligned).all())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GCN(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

gnn_model = GCN(num_features=5, num_classes=len(le_gnn.classes_)).to(device)
gnn_model.load_state_dict(torch.load(f'{checkpoint_dir_v4}/gnn_v4.pt', map_location=device))
gnn_model.eval()

gnn_proba_list = []
with torch.no_grad():
    for g in test_graphs_aligned:
        g = g.to(device)
        batch = torch.zeros(g.x.size(0), dtype=torch.long, device=device)
        out = gnn_model(g.x, g.edge_index, batch)
        gnn_proba_list.append(F.softmax(out, dim=1).cpu().numpy()[0])
gnn_proba = np.array(gnn_proba_list)

hybrid_proba = 0.4 * xgb_proba + 0.4 * lstm_proba + 0.2 * gnn_proba
y_pred_hybrid = hybrid_proba.argmax(axis=1)

y_test_binarized = label_binarize(y_test_aligned, classes=np.arange(len(le_gnn.classes_)))
roc_auc_macro = roc_auc_score(y_test_binarized, hybrid_proba, average='macro', multi_class='ovr')
roc_auc_par_classe = roc_auc_score(y_test_binarized, hybrid_proba, average=None, multi_class='ovr')

print("\n=== Metriques globales (Hybride XGBoost 0.4 + LSTM 0.4 + GNN 0.2) ===")
print(f"Accuracy         : {accuracy_score(y_test_aligned, y_pred_hybrid):.4f}")
print(f"Rappel (macro)   : {recall_score(y_test_aligned, y_pred_hybrid, average='macro'):.4f}")
print(f"F1-score (macro) : {f1_score(y_test_aligned, y_pred_hybrid, average='macro'):.4f}")
print(f"ROC-AUC (macro)  : {roc_auc_macro:.4f}")

print("\n=== ROC-AUC par classe ===")
with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    le_max = pickle.load(f)['le']
for cls, auc in zip(le_max.classes_, roc_auc_par_classe):
    print(f"  {cls:20s} : {auc:.4f}")

print("\n=== Rapport de classification complet (par classe) ===")
print(classification_report(y_test_aligned, y_pred_hybrid, target_names=le_max.classes_))

print("\n=== Matrice de confusion ===")
cm_hybrid = confusion_matrix(y_test_aligned, y_pred_hybrid)
cm_hybrid_df = pd.DataFrame(cm_hybrid, index=le_max.classes_, columns=le_max.classes_)
print(cm_hybrid_df)

np.savez(f'{checkpoint_dir_v4}/hybrid_proba_v4.npz',
         xgb_proba=xgb_proba, lstm_proba=lstm_proba, gnn_proba=gnn_proba, y_test=y_test_aligned)
cm_hybrid_df.to_csv(f'{checkpoint_dir_v4}/confusion_matrix_hybrid_v4.csv')
pd.Series(dict(zip(le_max.classes_, roc_auc_par_classe))).to_csv(f'{checkpoint_dir_v4}/roc_auc_hybrid_v4.csv')
print("\nProbabilites, matrice de confusion et ROC-AUC sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,115,116,117,118,153,155,157,158,159,160,161,162,163,164,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,247,248,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,86,90,91,92,93,94,96,98,99,100,101,102,103,104,105,107,108,109,110,111,112,113,114,115,116,117,153,155,157,158,159,160,161,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,250,251) have mixed types. Specify dtype option on import or set low_

Deauth: adresses MAC extraites


/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,90,91,92,98,99,100,101,102,103,104,105,107,108,109,110,111,115,119,120,135,137,140,144,146,148,149,151,167,168,178,180,183,184,185,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (71,73,77,78,79,80,81,82,83,84,85,86,90,91,92,93,94,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,122,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,191,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarn

Disas: adresses MAC extraites


/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,116,117,118,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,118,132,166,167,168,183,184,185,189,190,192,193,194,195,196,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (20,21,43,44,45,62,65,

... (17 outputs left out)

/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,87,90,91,92,93,98,99,100,101,102,103,104,105,107,108,109,110,111,115,118,166,167,168,172,178,180,181,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,242,243,244,247,248,249,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,90,91,92,93,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,116,117,118,166,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,249) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (71,73,77,78,79,80,86,93,112,113,114,116,117,118,166,167,168,183,184,185,189,190,

Evil_Twin: adresses MAC extraites


/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (23,24,48,50,51,81,82,83,84,85,86,90,91,92,93,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (82,84,85,86,87,90,91,92,93,94,96,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,249,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2860779969.py:24: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,87,90,91,92,93,97,98,99,100,101,102,103,

Website_spoofing: adresses MAC extraites
Graphes reconstruits : 154606
Verification alignement GNN et hybride (doit etre True) : True

=== Metriques globales (Hybride XGBoost 0.4 + LSTM 0.4 + GNN 0.2) ===
Accuracy         : 0.9403
Rappel (macro)   : 0.8688
F1-score (macro) : 0.8680
ROC-AUC (macro)  : 0.9962

=== ROC-AUC par classe ===
  Botnet               : 0.9986
  Deauth               : 0.9993
  Disas                : 0.9886
  Evil_Twin            : 0.9997
  Kr00k                : 0.9925
  Krack                : 0.9989
  Malware              : 0.9998
  Normal               : 0.9953
  Re-Assoc             : 0.9976
  Rogue_AP             : 0.9921
  SQL_Injection        : 0.9881
  SSDP                 : 1.0000
  SSH                  : 0.9967
  Website_spoofing     : 0.9997

=== Rapport de classification complet (par classe) ===
                  precision    recall  f1-score   support

          Botnet       0.96      0.93      0.95      1058
          Deauth       0.96      0.95     

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from tensorflow.keras import layers, models

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
protected_cols = ['Label', 'source_attack', 'source_file']
feature_cols = [c for c in df_encoded.columns if c not in protected_cols]

X_windows_seq, y_windows = [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_windows_seq.append(features[s:s + window_size])
        wl = labels[s:s + window_size]
        y_windows.append(attack_name if (wl != 'Normal').any() else 'Normal')
del df_encoded
gc.collect()

le_auto = LabelEncoder()
y_encoded = le_auto.fit_transform(y_windows)
X_seq = np.array(X_windows_seq, dtype=np.float32)
del X_windows_seq
gc.collect()

is_normal = (np.array(y_windows) == 'Normal')
X_normal = X_seq[is_normal]
X_attack = X_seq[~is_normal]
y_attack_labels = np.array(y_windows)[~is_normal]
print(f"Fenetres Normal disponibles : {len(X_normal)} | Fenetres attaque : {len(X_attack)}")

X_normal_train, X_normal_test = train_test_split(X_normal, test_size=0.2, random_state=42)

n_timesteps, n_features = X_normal_train.shape[1], X_normal_train.shape[2]
scaler = StandardScaler()
scaler.fit(X_normal_train.reshape(-1, n_features))

def scale(X):
    return scaler.transform(X.reshape(-1, n_features)).astype(np.float32).reshape(X.shape)

X_normal_train_s = scale(X_normal_train)
X_normal_test_s = scale(X_normal_test)
X_attack_s = scale(X_attack)

autoencoder = models.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),
    layers.LSTM(32, activation='tanh', return_sequences=False),
    layers.RepeatVector(n_timesteps),
    layers.LSTM(32, activation='tanh', return_sequences=True),
    layers.TimeDistributed(layers.Dense(n_features))
])
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

autoencoder.fit(
    X_normal_train_s, X_normal_train_s,
    validation_split=0.1, epochs=20, batch_size=128, verbose=1
)

def reconstruction_error(X_s):
    pred = autoencoder.predict(X_s, batch_size=256)
    return np.mean((X_s - pred) ** 2, axis=(1, 2))

err_normal_test = reconstruction_error(X_normal_test_s)
err_attack = reconstruction_error(X_attack_s)

y_true_global = np.concatenate([np.zeros(len(err_normal_test)), np.ones(len(err_attack))])
err_global = np.concatenate([err_normal_test, err_attack])
auc_global = roc_auc_score(y_true_global, err_global)
print(f"\n=== AUC globale (Normal vs toutes attaques) : {auc_global:.4f} ===")

print("\n=== AUC par attaque (Normal test vs cette attaque uniquement) ===")
auc_par_attaque = {}
for attack_name in le_auto.classes_:
    if attack_name == 'Normal':
        continue
    mask = y_attack_labels == attack_name
    if mask.sum() < 2:
        continue
    y_true = np.concatenate([np.zeros(len(err_normal_test)), np.ones(mask.sum())])
    err = np.concatenate([err_normal_test, err_attack[mask]])
    auc = roc_auc_score(y_true, err)
    auc_par_attaque[attack_name] = auc
    print(f"  {attack_name:20s} : AUC = {auc:.4f} (n={mask.sum()})")

autoencoder.save(f'{checkpoint_dir_v4}/autoencoder_v4.keras')
with open(f'{checkpoint_dir_v4}/scaler_autoencoder_v4.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'le': le_auto, 'feature_cols': feature_cols, 'err_normal_test': err_normal_test}, f)
pd.Series(auc_par_attaque).to_csv(f'{checkpoint_dir_v4}/autoencoder_auc_par_attaque_v4.csv')
print("\nModele, scaler et resultats sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fenetres Normal disponibles : 77944 | Fenetres attaque : 76662


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 20, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 20, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 20, 31)         │         1,023 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,535 (68.50 KB)

 Trainable params: 17,535 (68.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 0.8192 - val_loss: 0.7519
Epoch 2/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.7212 - val_loss: 0.6962
Epoch 3/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6791 - val_loss: 0.6659
Epoch 4/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6550 - val_loss: 0.6479
Epoch 5/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6367 - val_loss: 0.6336
Epoch 6/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6195 - val_loss: 0.6155
Epoch 7/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.6045 - val_loss: 0.6013
Epoch 8/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.5926 - val_loss: 0.5909
Epoch 9/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.5830 - val_loss: 0.5836
Epoch 10/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.5737 - val_loss: 0.5784
Epoch 11/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.5662 - val_loss: 0.5698
Epoch 12/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/ste

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import models as keras_models

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
protected_cols = ['Label', 'source_attack', 'source_file']
feature_cols = [c for c in df_encoded.columns if c not in protected_cols]

X_windows_seq, y_windows = [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_windows_seq.append(features[s:s + window_size])
        wl = labels[s:s + window_size]
        y_windows.append(attack_name if (wl != 'Normal').any() else 'Normal')
del df_encoded
gc.collect()

autoencoder = keras_models.load_model(f'{checkpoint_dir_v4}/autoencoder_v4.keras')
with open(f'{checkpoint_dir_v4}/scaler_autoencoder_v4.pkl', 'rb') as f:
    saved_auto = pickle.load(f)
scaler_auto = saved_auto['scaler']
err_normal_test = saved_auto['err_normal_test']

seuil_erreur = np.percentile(err_normal_test, 95)
print(f"Seuil d'erreur de reconstruction (95e percentile du Normal) : {seuil_erreur:.4f}")

data_hybrid = np.load(f'{checkpoint_dir_v4}/hybrid_proba_v4.npz')
y_test_aligned = data_hybrid['y_test']
hybrid_proba = 0.4 * data_hybrid['xgb_proba'] + 0.4 * data_hybrid['lstm_proba'] + 0.2 * data_hybrid['gnn_proba']
hybrid_confiance = hybrid_proba.max(axis=1)
hybrid_pred = hybrid_proba.argmax(axis=1)

all_idx = np.arange(len(y_windows))
y_windows_arr = np.array(y_windows)
with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    le_max = pickle.load(f)['le']
y_full_encoded_check = le_max.transform(y_windows_arr)
_, test_idx = train_test_split(all_idx, test_size=0.2, random_state=42, stratify=y_full_encoded_check)
print("Verification : etiquettes retrouvees identiques a celles de l'hybride (doit etre True) :",
      (y_full_encoded_check[test_idx] == y_test_aligned).all())

X_seq_test = np.array([X_windows_seq[i] for i in test_idx], dtype=np.float32)
del X_windows_seq
gc.collect()

n_timesteps, n_feat = X_seq_test.shape[1], X_seq_test.shape[2]
X_seq_test_scaled = scaler_auto.transform(X_seq_test.reshape(-1, n_feat)).astype(np.float32).reshape(X_seq_test.shape)

# Mesure du temps de diagnostic (TTD) : du calcul de l'erreur de reconstruction
# jusqu'a la decision du signal d'anomalie inedite, pour une fenetre moyenne
t0 = time.perf_counter()
pred_auto = autoencoder.predict(X_seq_test_scaled, batch_size=256)
err_test = np.mean((X_seq_test_scaled - pred_auto) ** 2, axis=(1, 2))
seuil_confiance = 0.5
anomalie_signalee = (err_test > seuil_erreur) & (hybrid_confiance < seuil_confiance)
t1 = time.perf_counter()

temps_total = t1 - t0
temps_par_fenetre_ms = (temps_total / len(X_seq_test)) * 1000
print(f"\n=== Temps de diagnostic (TTD) ===")
print(f"Temps total pour {len(X_seq_test)} fenetres : {temps_total:.3f} s")
print(f"Temps moyen par fenetre : {temps_par_fenetre_ms:.4f} ms")

print(f"\nFenetres signalees 'attaque potentiellement inedite' : {anomalie_signalee.sum()} / {len(anomalie_signalee)}")

noms_vrais = le_max.inverse_transform(y_test_aligned[anomalie_signalee])
print("\nRepartition des vraies etiquettes parmi les fenetres signalees :")
print(pd.Series(noms_vrais).value_counts())

noms_normal_signale = (le_max.inverse_transform(y_test_aligned[anomalie_signalee]) == 'Normal').sum()
print(f"\nDont fausses alertes (vrai Normal signale a tort) : {noms_normal_signale}")
print(f"Dont vraies attaques signalees : {anomalie_signalee.sum() - noms_normal_signale}")

accuracy_inchangee = (hybrid_pred == y_test_aligned).mean()
print(f"\nAccuracy globale de l'hybride (inchangee par ce signal supplementaire) : {accuracy_inchangee:.4f}")

results_pipeline = pd.DataFrame({
    'vraie_etiquette': le_max.inverse_transform(y_test_aligned),
    'prediction_hybride': le_max.inverse_transform(hybrid_pred),
    'confiance_hybride': hybrid_confiance,
    'erreur_autoencodeur': err_test,
    'anomalie_signalee': anomalie_signalee,
})
results_pipeline.to_csv(f'{checkpoint_dir_v4}/pipeline_deux_etages_v4.csv', index=False)

with open(f'{checkpoint_dir_v4}/ttd_pipeline_v4.pkl', 'wb') as f:
    pickle.dump({'temps_total_s': temps_total, 'temps_par_fenetre_ms': temps_par_fenetre_ms, 'n_fenetres': len(X_seq_test)}, f)
print("\nResultats et temps de diagnostic sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Seuil d'erreur de reconstruction (95e percentile du Normal) : 1.3066
Verification : etiquettes retrouvees identiques a celles de l'hybride (doit etre True) : True
121/121 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step

=== Temps de diagnostic (TTD) ===
Temps total pour 30922 fenetres : 1.593 s
Temps moyen par fenetre : 0.0515 ms

Fenetres signalees 'attaque potentiellement inedite' : 146 / 30922

Repartition des vraies etiquettes parmi les fenetres signalees :
Normal              64
Website_spoofing    13
Rogue_AP            11
Krack               10
Disas                7
Botnet               7
SSH                  7
SQL_Injection        6
Evil_Twin            6
Kr00k                5
Malware              4
SSDP                 3
Deauth               2
Re-Assoc             1
Name: count, dtype: int64

Dont fausses alertes (vrai Normal signale a tort) : 64
Dont vraies atta

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q shap

import gc, pickle, joblib, time
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

xgb_combo = joblib.load(f'{checkpoint_dir_v4}/xgb_combo_v4.joblib')
with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']
feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')

X_windows_max, y_windows = [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_windows_max.append(features[s:s + window_size].max(axis=0))
        wl = labels[s:s + window_size]
        y_windows.append(attack_name if (wl != 'Normal').any() else 'Normal')
del df_encoded
gc.collect()

y_encoded = le_max.transform(y_windows)
X_max = np.array(X_windows_max, dtype=np.float32)

_, X_test, _, y_test = train_test_split(
    X_max, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print("Verification : accuracy retrouvee sur le test (attendu 0.8729) :",
      round(accuracy_score(y_test, xgb_combo.predict(X_test)), 4))

rng = np.random.RandomState(42)
sample_idx = []
for c in range(len(le_max.classes_)):
    idx_c = np.where(y_test == c)[0]
    take = min(300, len(idx_c))
    sample_idx.extend(rng.choice(idx_c, size=take, replace=False))
sample_idx = np.array(sample_idx)
X_sample = X_test[sample_idx]
y_sample = y_test[sample_idx]
print("Taille de l'echantillon explique :", X_sample.shape)

# === Temps de calcul SHAP, mesure directe ===
t0 = time.perf_counter()
explainer = shap.TreeExplainer(xgb_combo)
sv = explainer.shap_values(X_sample)
t1 = time.perf_counter()
if isinstance(sv, list):
    sv = np.stack(sv, axis=-1)
temps_shap_total = t1 - t0
temps_shap_par_fenetre_ms = (temps_shap_total / len(X_sample)) * 1000
print(f"\n=== Temps de calcul SHAP ===")
print(f"Temps total pour {len(X_sample)} fenetres : {temps_shap_total:.3f} s")
print(f"Temps moyen par fenetre : {temps_shap_par_fenetre_ms:.4f} ms")

# === Stabilite : perturbation legere de l'entree, recalcul, comparaison ===
rng_pert = np.random.RandomState(0)
noise = rng_pert.normal(0, 0.01, X_sample.shape).astype(np.float32) * (np.abs(X_sample).max(axis=0) + 1e-6)
X_sample_perturbed = X_sample + noise
sv_perturbed = explainer.shap_values(X_sample_perturbed)
if isinstance(sv_perturbed, list):
    sv_perturbed = np.stack(sv_perturbed, axis=-1)

idx_n = np.arange(len(X_sample))
pred0 = xgb_combo.predict_proba(X_sample).argmax(axis=1)
sv_pred = sv[idx_n, :, pred0]
sv_pred_perturbed = sv_perturbed[idx_n, :, pred0]
diff_norm = np.linalg.norm(sv_pred - sv_pred_perturbed, axis=1)
orig_norm = np.linalg.norm(sv_pred, axis=1) + 1e-8
stabilite = 1 - np.mean(diff_norm / orig_norm)
print(f"\n=== Stabilite SHAP (perturbation legere, 1% de bruit) ===")
print(f"Score de stabilite (1 = parfaitement stable) : {stabilite:.4f}")

# === Fidelite : masquage des caracteristiques desgnees vs hasard ===
baseline = np.median(X_test, axis=0)
p0 = xgb_combo.predict_proba(X_sample)[idx_n, pred0]
order_shap = np.argsort(-sv_pred, axis=1)

def masked(X, order, k):
    Xm = X.copy()
    cols = order[:, :k]
    rows = np.arange(len(X))[:, None]
    Xm[rows, cols] = baseline[cols]
    return Xm

R = 5
rng_r = np.random.RandomState(1)
orders_rand = [rng_r.random_sample(X_sample.shape).argsort(axis=1) for _ in range(R)]

print("\n=== Fidelite SHAP (k=1, 5, 10) ===")
for k in [1, 5, 10]:
    ps = xgb_combo.predict_proba(masked(X_sample, order_shap, k))
    drop_s = (p0 - ps[idx_n, pred0]).mean()
    drop_r = np.mean([(p0 - xgb_combo.predict_proba(masked(X_sample, o, k))[idx_n, pred0]).mean() for o in orders_rand])
    print(f"  k={k:2d} : chute SHAP = {drop_s:.4f} | chute hasard = {drop_r:.4f}")

# === Cohesion : recouvrement du top-5 entre fenetres d'une meme classe ===
print("\n=== Coherence des contributions (recouvrement top-5 intra-classe) ===")
coherence_par_classe = {}
for c, cls_name in enumerate(le_max.classes_):
    mask = y_sample == c
    if mask.sum() < 2:
        continue
    imp_c = np.abs(sv[mask][:, :, c])
    top5_sets = [set(np.argsort(-row)[:5]) for row in imp_c]
    n = len(top5_sets)
    overlaps = []
    for i in range(min(n, 50)):
        for j in range(i + 1, min(n, 50)):
            overlaps.append(len(top5_sets[i] & top5_sets[j]) / 5)
    coherence = np.mean(overlaps) if overlaps else np.nan
    coherence_par_classe[cls_name] = coherence
    print(f"  {cls_name:20s} : {coherence:.4f}")

results_shap_metrics = pd.DataFrame({
    'metrique': ['temps_par_fenetre_ms', 'stabilite'],
    'valeur': [temps_shap_par_fenetre_ms, stabilite]
})
results_shap_metrics.to_csv(f'{checkpoint_dir_v4}/shap_metriques_globales_v4.csv', index=False)
pd.Series(coherence_par_classe).to_csv(f'{checkpoint_dir_v4}/shap_coherence_par_classe_v4.csv')
print("\nMetriques SHAP sauvegardees.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verification : accuracy retrouvee sur le test (attendu 0.8729) : 0.8729
Taille de l'echantillon explique : (3848, 31)

=== Temps de calcul SHAP ===
Temps total pour 3848 fenetres : 7.177 s
Temps moyen par fenetre : 1.8652 ms

=== Stabilite SHAP (perturbation legere, 1% de bruit) ===
Score de stabilite (1 = parfaitement stable) : -0.5734

=== Fidelite SHAP (k=1, 5, 10) ===
  k= 1 : chute SHAP = 0.4040 | chute hasard = 0.0363
  k= 5 : chute SHAP = 0.7342 | chute hasard = 0.1816
  k=10 : chute SHAP = 0.7592 | chute hasard = 0.3547

=== Coherence des contributions (recouvrement top-5 intra-classe) ===
  Botnet               : 0.6833
  Deauth               : 0.5293
  Disas                : 0.6911
  Evil_Twin            : 0.7451
  Kr00k                : 0.6584
  Krack                : 0.5455
  Malware              : 0.8057
  Normal               : 0.4784
  Re-Assoc

In [ ]:
from scipy.stats import spearmanr

correlations = [spearmanr(sv_pred[i], sv_pred_perturbed[i])[0] for i in range(len(sv_pred))]
stabilite_spearman = np.nanmean(correlations)

top5_orig = [set(np.argsort(-np.abs(sv_pred[i]))[:5]) for i in range(len(sv_pred))]
top5_pert = [set(np.argsort(-np.abs(sv_pred_perturbed[i]))[:5]) for i in range(len(sv_pred_perturbed))]
recouvrement_top5 = np.mean([len(a & b) / 5 for a, b in zip(top5_orig, top5_pert)])

print(f"Stabilite (correlation de Spearman) : {stabilite_spearman:.4f}")
print(f"Stabilite (recouvrement du top-5)   : {recouvrement_top5:.4f}")

pd.DataFrame({'metrique': ['stabilite_spearman', 'recouvrement_top5'],
              'valeur': [stabilite_spearman, recouvrement_top5]}).to_csv(
    f'{checkpoint_dir_v4}/shap_stabilite_corrigee_v4.csv', index=False)
print("Stabilite corrigee sauvegardee.")

Stabilite (correlation de Spearman) : 0.2907
Stabilite (recouvrement du top-5)   : 0.5607
Stabilite corrigee sauvegardee.


In [ ]:
# Reutilise X_sample, sv_pred, explainer deja calcules dans la cellule precedente
binary_mask = np.array([X_test[:, i].nunique() <= 2 if hasattr(X_test[:, i], 'nunique')
                         else len(np.unique(X_test[:, i])) <= 2 for i in range(X_test.shape[1])])
print(f"Caracteristiques binaires detectees : {binary_mask.sum()} / {len(binary_mask)}")
print([feature_cols[i] for i in range(len(feature_cols)) if binary_mask[i]])

rng_pert2 = np.random.RandomState(0)
noise2 = rng_pert2.normal(0, 0.01, X_sample.shape).astype(np.float32) * (np.abs(X_sample).max(axis=0) + 1e-6)
noise2[:, binary_mask] = 0.0

X_sample_perturbed2 = X_sample + noise2
sv_perturbed2 = explainer.shap_values(X_sample_perturbed2)
if isinstance(sv_perturbed2, list):
    sv_perturbed2 = np.stack(sv_perturbed2, axis=-1)
sv_pred_perturbed2 = sv_perturbed2[idx_n, :, pred0]

from scipy.stats import spearmanr
correlations2 = [spearmanr(sv_pred[i], sv_pred_perturbed2[i])[0] for i in range(len(sv_pred))]
stabilite_spearman2 = np.nanmean(correlations2)

top5_orig2 = [set(np.argsort(-np.abs(sv_pred[i]))[:5]) for i in range(len(sv_pred))]
top5_pert2 = [set(np.argsort(-np.abs(sv_pred_perturbed2[i]))[:5]) for i in range(len(sv_pred_perturbed2))]
recouvrement_top5_2 = np.mean([len(a & b) / 5 for a, b in zip(top5_orig2, top5_pert2)])

print(f"\nStabilite corrigee, perturbation uniquement continue :")
print(f"Correlation de Spearman : {stabilite_spearman2:.4f}")
print(f"Recouvrement du top-5   : {recouvrement_top5_2:.4f}")

pd.DataFrame({'metrique': ['stabilite_spearman_continu', 'recouvrement_top5_continu'],
              'valeur': [stabilite_spearman2, recouvrement_top5_2]}).to_csv(
    f'{checkpoint_dir_v4}/shap_stabilite_corrigee2_v4.csv', index=False)
print("Stabilite corrigee (v2) sauvegardee.")

Caracteristiques binaires detectees : 19 / 31
['wlan.fc.frag', 'wlan.fc.order', 'wlan.fc.moredata', 'wlan.fc.protected', 'wlan.fc.pwrmgt', 'wlan.fc.retry', 'tcp.flags.syn', 'tcp.flags.fin', 'tcp.flags.push', 'tcp.flags.reset', 'wlan.fc.ds_0x00000000', 'wlan.fc.ds_0x00000001', 'wlan.fc.ds_0x00000002', 'wlan.fc.ds_0x00000003', 'llc_llc', 'llc_llc-llc', 'llc_llc-llc-llc', 'llc_<NA>', 'tcp.analysis_clean']

Stabilite corrigee, perturbation uniquement continue :
Correlation de Spearman : 0.5349
Recouvrement du top-5   : 0.6782
Stabilite corrigee (v2) sauvegardee.


In [ ]:
# Reutilise X_sample, sv_pred, explainer, binary_mask deja calcules
stds = X_test.std(axis=0)
rng_pert3 = np.random.RandomState(0)
noise3 = rng_pert3.normal(0, 0.05, X_sample.shape).astype(np.float32) * stds
noise3[:, binary_mask] = 0.0

X_sample_perturbed3 = X_sample + noise3
sv_perturbed3 = explainer.shap_values(X_sample_perturbed3)
if isinstance(sv_perturbed3, list):
    sv_perturbed3 = np.stack(sv_perturbed3, axis=-1)
sv_pred_perturbed3 = sv_perturbed3[idx_n, :, pred0]

from scipy.stats import spearmanr
correlations3 = [spearmanr(sv_pred[i], sv_pred_perturbed3[i])[0] for i in range(len(sv_pred))]
stabilite_spearman3 = np.nanmean(correlations3)

top5_orig3 = [set(np.argsort(-np.abs(sv_pred[i]))[:5]) for i in range(len(sv_pred))]
top5_pert3 = [set(np.argsort(-np.abs(sv_pred_perturbed3[i]))[:5]) for i in range(len(sv_pred_perturbed3))]
recouvrement_top5_3 = np.mean([len(a & b) / 5 for a, b in zip(top5_orig3, top5_pert3)])

print(f"Bruit : 5% de l'ecart-type de chaque caracteristique continue")
print(f"Correlation de Spearman : {stabilite_spearman3:.4f}")
print(f"Recouvrement du top-5   : {recouvrement_top5_3:.4f}")

pd.DataFrame({'metrique': ['stabilite_spearman_std', 'recouvrement_top5_std'],
              'valeur': [stabilite_spearman3, recouvrement_top5_3]}).to_csv(
    f'{checkpoint_dir_v4}/shap_stabilite_corrigee3_v4.csv', index=False)
print("Stabilite corrigee (v3) sauvegardee.")

Bruit : 5% de l'ecart-type de chaque caracteristique continue
Correlation de Spearman : 0.5444
Recouvrement du top-5   : 0.6830
Stabilite corrigee (v3) sauvegardee.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

data = np.load(f'{checkpoint_dir_v4}/hybrid_proba_v4.npz')
P = {'XGBoost': data['xgb_proba'], 'LSTM': data['lstm_proba'], 'GNN': data['gnn_proba']}
y_test = data['y_test']
with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    classes = pickle.load(f)['le'].classes_
W = {'XGBoost': 0.4, 'LSTM': 0.4, 'GNN': 0.2}

hybrid = sum(W[m] * P[m] for m in P)
pred = hybrid.argmax(axis=1)
correct = pred == y_test

flip = {}
for m in P:
    without = sum(W[k] * P[k] for k in P if k != m)
    flip[m] = without.argmax(axis=1) != pred

flip_table = {}
for c, name in enumerate(classes):
    mask = (y_test == c) & correct
    flip_table[name] = {m: round(float(flip[m][mask].mean() * 100), 1) for m in P}

print("=== Pourcentage de fenetres dont la decision change si on retire ce modele ===")
print(pd.DataFrame(flip_table).T)

print("\n=== Valeur de la combinaison ===")
for m in P:
    alone_ok = P[m].argmax(axis=1) == y_test
    print(f"{m:8s} : hybride juste alors que {m} seul est faux = {int((~alone_ok & correct).sum()):5d} "
          f"| {m} seul juste alors que l'hybride est faux = {int((alone_ok & ~correct).sum()):5d}")

pd.DataFrame(flip_table).T.to_csv(f'{checkpoint_dir_v4}/decomposition_hybride_ablation_v4.csv')
print("\nTableau sauvegarde.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Pourcentage de fenetres dont la decision change si on retire ce modele ===
                  XGBoost  LSTM  GNN
Botnet                3.3   7.3  0.9
Deauth                0.5  12.4  0.2
Disas                 3.7  12.3  1.3
Evil_Twin             0.1   1.9  0.3
Kr00k                 5.5   9.6  0.5
Krack                 1.9   5.2  0.7
Malware               5.2   0.5  0.2
Normal                3.4   5.7  1.9
Re-Assoc              0.0  21.9  0.0
Rogue_AP             11.4  41.8  0.0
SQL_Injection        29.3  43.1  0.0
SSDP                  0.2   0.0  0.0
SSH                   6.7  17.6  3.5
Website_spoofing      3.2   1.7  0.2

=== Valeur de la combinaison ===
XGBoost  : hybride juste alors que XGBoost seul est faux =  2373 | XGBoost seul juste alors que l'hybride est faux =   291
LSTM     : hybride juste alors que LSTM seul est faux =  1387 | LSTM seul juste 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             confusion_matrix, precision_recall_fscore_support)

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

data = np.load(f'{checkpoint_dir_v4}/hybrid_proba_v4.npz')
P_x, P_l, P_g = data['xgb_proba'], data['lstm_proba'], data['gnn_proba']
y_test = data['y_test']
with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    classes = pickle.load(f)['le'].classes_
n_cls = len(classes)
normal_idx = list(classes).index('Normal')
attack_mask = np.arange(n_cls) != normal_idx
focus = ['Rogue_AP', 'SQL_Injection', 'SSH']

configs = {
    'Actuel (0.4/0.4/0.2)': (0.4, 0.4, 0.2),
    'Sans GNN (0.5/0.5/0)': (0.5, 0.5, 0.0),
}
rows = []
preds = {}
for name, (wx, wl, wg) in configs.items():
    pred = (wx * P_x + wl * P_l + wg * P_g).argmax(axis=1)
    preds[name] = pred
    cm = confusion_matrix(y_test, pred, labels=range(n_cls))
    _, _, f1_cls, _ = precision_recall_fscore_support(y_test, pred, labels=range(n_cls), zero_division=0)
    row = {
        'configuration': name,
        'accuracy': accuracy_score(y_test, pred),
        'rappel_macro': recall_score(y_test, pred, average='macro'),
        'f1_macro': f1_score(y_test, pred, average='macro'),
        'fausses_alertes': int(cm[normal_idx].sum() - cm[normal_idx, normal_idx]),
        'attaques_vues_Normal': int(cm[attack_mask, normal_idx].sum()),
    }
    for c in focus:
        row[f'F1_{c}'] = f1_cls[list(classes).index(c)]
    rows.append(row)

res = pd.DataFrame(rows).set_index('configuration')
print("=== Comparaison avec et sans GNN ===\n")
print(res.round(4).T)

only_actuel = int(((preds['Actuel (0.4/0.4/0.2)'] == y_test) & (preds['Sans GNN (0.5/0.5/0)'] != y_test)).sum())
only_sans = int(((preds['Actuel (0.4/0.4/0.2)'] != y_test) & (preds['Sans GNN (0.5/0.5/0)'] == y_test)).sum())
print(f"\nJuste avec le GNN, faux sans lui : {only_actuel}")
print(f"Faux avec le GNN, juste sans lui : {only_sans}")

print("\n=== Balayage du frein (sans GNN, frein sur Normal) ===")
s2 = 0.5 * P_x + 0.5 * P_l
sweep_rows = []
for k in np.round(np.arange(1.0, 2.01, 0.1), 2):
    s = s2.copy()
    s[:, normal_idx] *= k
    pred_k = s.argmax(axis=1)
    cm_k = confusion_matrix(y_test, pred_k, labels=range(n_cls))
    sweep_rows.append({
        'k': k,
        'accuracy': accuracy_score(y_test, pred_k),
        'fausses_alertes': int(cm_k[normal_idx].sum() - cm_k[normal_idx, normal_idx]),
        'attaques_vues_Normal': int(cm_k[attack_mask, normal_idx].sum()),
    })
sweep = pd.DataFrame(sweep_rows).set_index('k')
print(sweep.round(4))

print(f"\nMeilleure accuracy atteinte par le frein : {sweep['accuracy'].max():.4f}")
print(f"Accuracy avec GNN (reference) : {res.loc['Actuel (0.4/0.4/0.2)', 'accuracy']:.4f}")

res.to_csv(f'{checkpoint_dir_v4}/comparaison_poids_hybride_v4.csv')
sweep.to_csv(f'{checkpoint_dir_v4}/balayage_frein_normal_v4.csv')
print("\nTableaux sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Comparaison avec et sans GNN ===

configuration         Actuel (0.4/0.4/0.2)  Sans GNN (0.5/0.5/0)
accuracy                            0.9403                0.9330
rappel_macro                        0.8688                0.8824
f1_macro                            0.8680                0.8559
fausses_alertes                   384.0000              667.0000
attaques_vues_Normal              500.0000              418.0000
F1_Rogue_AP                         0.5962                0.5863
F1_SQL_Injection                    0.5133                0.4781
F1_SSH                              0.8113                0.7222

Juste avec le GNN, faux sans lui : 360
Faux avec le GNN, juste sans lui : 134

=== Balayage du frein (sans GNN, frein sur Normal) ===
     accuracy  fausses_alertes  attaques_vues_Normal
k                                                   
1.0    

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']
feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
X_windows_max, y_windows = [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_windows_max.append(features[s:s + window_size].max(axis=0))
        wl = labels[s:s + window_size]
        y_windows.append(attack_name if (wl != 'Normal').any() else 'Normal')
del df_encoded
gc.collect()

y_encoded = le_max.transform(y_windows)
X_max = np.array(X_windows_max, dtype=np.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X_max, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

time_cols = [c for c in ['tcp.time_relative'] if c in feature_cols]
port_cols = [c for c in ['tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport'] if c in feature_cols]
print("Colonnes retirees :", time_cols, "puis", port_cols)

sets = {'Toutes': [], 'Sans time_relative ni ports': time_cols + port_cols}
rows = []
for name, removed in sets.items():
    keep = [i for i, c in enumerate(feature_cols) if c not in removed]
    Xtr, Xte = X_train[:, keep], X_test[:, keep]
    Xtr_res, ytr_res = SMOTE(random_state=42, k_neighbors=5).fit_resample(Xtr, y_train)
    model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          eval_metric='mlogloss', random_state=42)
    model.fit(Xtr_res, ytr_res)
    pred = model.predict(Xte)
    rows.append({
        'jeu': name, 'nb_caracteristiques': len(keep),
        'accuracy': accuracy_score(y_test, pred),
        'rappel_macro': recall_score(y_test, pred, average='macro'),
        'f1_macro': f1_score(y_test, pred, average='macro'),
    })
    print(f"[{name}] termine.")
    del Xtr, Xte, Xtr_res, ytr_res, model
    gc.collect()

summary = pd.DataFrame(rows).set_index('jeu')
print("\n=== Verification : 'Toutes' doit redonner environ 0.8729 / 0.7951 / 0.7607 ===")
print(summary.round(4))

summary.to_csv(f'{checkpoint_dir_v4}/dependance_colonnes_globale_v4.csv')
print("\nTableau sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colonnes retirees : ['tcp.time_relative'] puis ['tcp.srcport', 'tcp.dstport']
[Toutes] termine.
[Sans time_relative ni ports] termine.

=== Verification : 'Toutes' doit redonner environ 0.8729 / 0.7951 / 0.7607 ===
                             nb_caracteristiques  accuracy  rappel_macro  \
jeu                                                                        
Toutes                                        31    0.8729        0.7951   
Sans time_relative ni ports                   28    0.7926        0.6755   

                             f1_macro  
jeu                                    
Toutes                         0.7607  
Sans time_relative ni ports    0.6438  

Tableau sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']
feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
X_windows_max, y_windows, file_of_window = [], [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_windows_max.append(features[s:s + window_size].max(axis=0))
        wl = labels[s:s + window_size]
        y_windows.append(attack_name if (wl != 'Normal').any() else 'Normal')
        file_of_window.append(file_id)
del df_encoded
gc.collect()

y_encoded = le_max.transform(y_windows)
X_max = np.array(X_windows_max, dtype=np.float32)
file_of_window = np.array(file_of_window)

files_by_attack = {}
for f in pd.unique(file_of_window):
    files_by_attack.setdefault(f.split('__')[0], []).append(f)
fold_of_file = {f: i for fl in files_by_attack.values() for i, f in enumerate(fl)}
fold_of_window = np.array([fold_of_file[f] for f in file_of_window])
n_folds = max(fold_of_file.values()) + 1

for k in range(n_folds):
    path = f'{checkpoint_dir_v4}/cv_fichiers_pli{k}_v4.npz'
    if os.path.exists(path):
        print(f"Pli {k} deja fait, ignore.")
        continue
    test_mask = fold_of_window == k
    train_mask = ~test_mask
    Xtr, ytr = X_max[train_mask], y_encoded[train_mask]
    Xtr_res, ytr_res = SMOTE(random_state=42, k_neighbors=5).fit_resample(Xtr, ytr)
    model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          eval_metric='mlogloss', random_state=42)
    model.fit(Xtr_res, ytr_res)
    pred = model.predict(X_max[test_mask])
    np.savez(path, y_true=y_encoded[test_mask], y_pred=pred)
    print(f"Pli {k} termine ({int(test_mask.sum())} fenetres de test).")
    del Xtr, ytr, Xtr_res, ytr_res, model
    gc.collect()

data = [np.load(f'{checkpoint_dir_v4}/cv_fichiers_pli{k}_v4.npz') for k in range(n_folds)]
y_true = np.concatenate([d['y_true'] for d in data])
y_pred = np.concatenate([d['y_pred'] for d in data])

print("\n=== Generalisation a des fichiers jamais vus (resultats regroupes) ===")
print(f"Accuracy       : {accuracy_score(y_true, y_pred):.4f}")
print(f"Rappel (macro) : {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1-score (macro) : {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nRappel : decoupage par fenetres = accuracy 0.8729, rappel macro 0.7951, F1 macro 0.7607")

print("\n=== Accuracy par pli ===")
for k, d in enumerate(data):
    print(f"  Pli {k} : {accuracy_score(d['y_true'], d['y_pred']):.4f}")

pd.DataFrame({'y_true': y_true, 'y_pred': y_pred}).to_csv(f'{checkpoint_dir_v4}/cv_fichiers_resultats_v4.csv', index=False)
print("\nResultats sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pli 0 termine (32247 fenetres de test).
Pli 1 termine (32232 fenetres de test).
Pli 2 termine (30356 fenetres de test).
Pli 3 termine (31428 fenetres de test).
Pli 4 termine (28343 fenetres de test).

=== Generalisation a des fichiers jamais vus (resultats regroupes) ===
Accuracy       : 0.8130
Rappel (macro) : 0.6651
F1-score (macro) : 0.6654

Rappel : decoupage par fenetres = accuracy 0.8729, rappel macro 0.7951, F1 macro 0.7607

=== Accuracy par pli ===
  Pli 0 : 0.8415
  Pli 1 : 0.8256
  Pli 2 : 0.8006
  Pli 3 : 0.7982
  Pli 4 : 0.7958

Resultats sauvegardes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time
from collections import Counter, defaultdict
import pandas as pd

base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}

addr_cols = ['wlan.ta', 'wlan.sa', 'wlan.bssid', 'ip.src']
wanted = set(addr_cols + ['Label'])

att_cnt, nor_cnt, top_per_file = defaultdict(Counter), defaultdict(Counter), defaultdict(list)
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        df = read_csv_safe(os.path.join(folder_path, f), usecols=lambda c: c in wanted)
        is_att = df['Label'] != 'Normal'
        if is_att.sum() == 0:
            del df; gc.collect(); continue
        for col in addr_cols:
            if col not in df.columns:
                continue
            a = df.loc[is_att, col].dropna().value_counts()
            n = df.loc[~is_att, col].dropna().value_counts()
            att_cnt[(attack_name, col)].update(a.to_dict())
            nor_cnt[(attack_name, col)].update(n.to_dict())
            if len(a):
                top_per_file[(attack_name, col)].append(a.index[0])
        del df; gc.collect()
    print(f'{attack_name}: termine')

rows = []
for (attack_name, col), a_counter in att_cnt.items():
    if not a_counter:
        continue
    total = sum(a_counter.values())
    top_addr, top_n = a_counter.most_common(1)[0]
    n_norm_top = nor_cnt[(attack_name, col)].get(top_addr, 0)
    tops = top_per_file[(attack_name, col)]
    rows.append({
        'attaque': attack_name, 'champ': col, 'lignes_attaque': total,
        'adresses_distinctes': len(a_counter), 'adresse_principale': top_addr,
        'part_lignes_attaque_pourcent': round(top_n / total * 100, 1),
        'purete_pourcent': round(top_n / (top_n + n_norm_top) * 100, 1),
        'fichiers_ou_principale': f"{sum(1 for t in tops if t == top_addr)}/{len(tops)}",
    })

res = pd.DataFrame(rows).sort_values(['attaque', 'champ']).reset_index(drop=True)
pd.set_option('display.width', 250)
print("\n=== Adresses dans les lignes etiquetees comme attaque ===")
print(res.to_string(index=False))

res.to_csv(f'{checkpoint_dir_v4}/verite_terrain_adresses_v4.csv', index=False)
print("\nTableau sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Deauth: termine


/tmp/ipykernel_7865/2197464362.py:14: DtypeWarning: Columns (83) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_7865/2197464362.py:14: DtypeWarning: Columns (83) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)


Disas: termine
Re-Assoc: termine
Rogue_AP: termine
Krack: termine
Kr00k: termine
SSH: termine
Botnet: termine
Malware: termine
SQL_Injection: termine
SSDP: termine
Evil_Twin: termine


/tmp/ipykernel_7865/2197464362.py:14: DtypeWarning: Columns (83) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)


Website_spoofing: termine

=== Adresses dans les lignes etiquetees comme attaque ===
         attaque      champ  lignes_attaque  adresses_distinctes adresse_principale  part_lignes_attaque_pourcent  purete_pourcent fichiers_ou_principale
          Botnet     ip.src           28494                   10      192.168.2.248                          57.6            100.0                    3/5
          Botnet wlan.bssid           28494                    1  0c:9d:92:54:fe:34                         100.0             23.9                    5/5
          Botnet    wlan.sa           28494                   10  04:ed:33:e0:24:82                          57.6             99.0                    3/5
          Botnet    wlan.ta           28494                    7  04:ed:33:e0:24:82                          52.9             63.6                    2/5
          Deauth wlan.bssid           25831                    1  0c:9d:92:54:fe:34                         100.0             23.5               

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch_geometric

import gc, time, pickle, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool
from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.loader import DataLoader as PyGDataLoader
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
print("Determinisme complet active.")

base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}
mac_cols = ['wlan.sa', 'wlan.da', 'wlan.ta', 'wlan.ra', 'wlan.bssid']
role_cols_map = {'wlan.sa': 0, 'wlan.da': 1, 'wlan.ta': 2, 'wlan.ra': 3, 'wlan.bssid': 4}

mac_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        df_raw = read_csv_safe(os.path.join(folder_path, f)).reset_index(drop=True)
        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw; gc.collect(); continue
        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        cols_present = [c for c in mac_cols if c in df_raw.columns] + ['Label']
        df_mac_slice = df_raw.iloc[start:end][cols_present].reset_index(drop=True)
        file_id = f'{attack_name}__{f}'
        df_mac_slice['source_file'] = file_id
        mac_slices[file_id] = df_mac_slice
        del df_raw; gc.collect()
    print(f'{attack_name}: adresses MAC extraites')

df_mac_combined = pd.concat(mac_slices.values(), ignore_index=True)
del mac_slices
gc.collect()

with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'rb') as f:
    le_gnn = pickle.load(f)['le']

graph_data_list, graph_node_macs, graph_labels = [], [], []
for file_id, group in df_mac_combined.groupby('source_file', sort=False):
    mac_cols_present = [c for c in mac_cols if c in group.columns]
    attack_label = file_id.split('__')[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window_df = group.iloc[s:s + window_size]
        edges = set()
        role_counts = {}
        for _, row in window_df.iterrows():
            present = [(c, row[c]) for c in mac_cols_present if pd.notna(row[c])]
            addrs = [a for _, a in present]
            for col, mac in present:
                if mac not in role_counts:
                    role_counts[mac] = [0, 0, 0, 0, 0]
                role_counts[mac][role_cols_map[col]] += 1
            if len(addrs) >= 2:
                edges.add(tuple(sorted(addrs[:2])))
        node_list = sorted(role_counts.keys()) if role_counts else ['isole']
        if 'isole' in node_list:
            role_counts['isole'] = [0, 0, 0, 0, 0]
        node_idx = {mac: idx for idx, mac in enumerate(node_list)}
        edge_index_list = []
        for a, b in edges:
            edge_index_list.append([node_idx[a], node_idx[b]])
            edge_index_list.append([node_idx[b], node_idx[a]])
        x = torch.tensor([role_counts[mac] for mac in node_list], dtype=torch.float)
        edge_index = (torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
                      if edge_index_list else torch.zeros((2, 0), dtype=torch.long))
        window_labels = window_df['Label'].values
        label_name = attack_label if (window_labels != 'Normal').any() else 'Normal'
        graph_data_list.append(Data(x=x, edge_index=edge_index))
        graph_node_macs.append(node_list)
        graph_labels.append(label_name)
del df_mac_combined
gc.collect()

y_gnn_encoded = le_gnn.transform(graph_labels)
for data, y_val in zip(graph_data_list, y_gnn_encoded):
    data.y = torch.tensor([y_val])

graph_indices = np.arange(len(graph_data_list))
train_idx_g, test_idx_g = train_test_split(graph_indices, test_size=0.2, random_state=SEED, stratify=y_gnn_encoded)
print("Graphes :", len(graph_data_list), "| Test :", len(test_idx_g))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GCN_Mean(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GCN_Sum(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_add_pool(x, batch)
        return self.lin(x)

gnn_mean = GCN_Mean(num_features=5, num_classes=len(le_gnn.classes_)).to(device)
gnn_mean.load_state_dict(torch.load(f'{checkpoint_dir_v4}/gnn_v4.pt', map_location=device))
gnn_mean.eval()
print("GNN (pooling par moyenne) recharge.")

graph_data_list = [d.cpu() for d in graph_data_list]

torch.manual_seed(SEED)
train_graphs_sum = [graph_data_list[i] for i in range(len(graph_data_list)) if i not in set(test_idx_g)]
gnn_sum = GCN_Sum(num_features=5, num_classes=len(le_gnn.classes_)).to(device)
optimizer = torch.optim.Adam(gnn_sum.parameters(), lr=0.01)
g = torch.Generator().manual_seed(SEED)
train_loader_sum = PyGDataLoader(train_graphs_sum, batch_size=128, shuffle=True, generator=g)
for epoch in range(15):
    gnn_sum.train()
    for batch in train_loader_sum:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = gnn_sum(batch.x, batch.edge_index, batch.batch)
        loss = F.cross_entropy(out, batch.y)
        loss.backward()
        optimizer.step()
print("GNN a pooling par somme reentraine (deterministe).")
torch.save(gnn_sum.state_dict(), f'{checkpoint_dir_v4}/gnn_sum_v4.pt')

def proba_model(model, data, cls_idx):
    with torch.no_grad():
        d = data.to(device)
        batch = torch.zeros(d.x.size(0), dtype=torch.long, device=device)
        out = model(d.x, d.edge_index, batch)
        return F.softmax(out, dim=1)[0, cls_idx].item()

reference = {
    'Malware': '00:0c:29:cf:08:aa', 'SSH': '00:0c:29:1d:70:f5',
    'Botnet': '04:ed:33:e0:24:82', 'Krack': '94:e9:79:82:c5:77',
    'Website_spoofing': '04:ed:33:e0:24:82', 'SSDP': '04:ed:33:e0:24:82',
}
reference_sum = {'Evil_Twin': '0c:9d:92:54:fe:35'}

torch.manual_seed(SEED)
explainer = Explainer(
    model=gnn_mean, algorithm=GNNExplainer(epochs=100),
    explanation_type='model', node_mask_type='object', edge_mask_type=None,
    model_config=dict(mode='multiclass_classification', task_level='graph', return_type='raw'),
)

rng = np.random.RandomState(SEED)
rows = []
for label in list(reference.keys()) + list(reference_sum.keys()):
    pool = [i for i in test_idx_g if graph_labels[i] == label]
    sample = rng.choice(pool, size=min(300, len(pool)), replace=False)
    ref_mac = reference.get(label) or reference_sum.get(label)
    model_use = gnn_sum if label in reference_sum else gnn_mean
    cls_idx = int(le_gnn.transform([label])[0])

    for idx in sample:
        data = graph_data_list[idx]
        macs = graph_node_macs[idx]
        n_nodes = data.x.size(0)
        if n_nodes < 2:
            continue
        all_proba = [proba_model(model_use, data, c) for c in range(len(le_gnn.classes_))]
        pred0 = le_gnn.inverse_transform([int(np.argmax(all_proba))])[0]
        if pred0 != label:
            continue

        if label in reference_sum:
            p0 = all_proba[cls_idx]
            scores = []
            for node in range(n_nodes):
                keep = [i for i in range(n_nodes) if i != node]
                remap = {old: new for new, old in enumerate(keep)}
                x2 = data.x[keep]
                mask_edge = [(e0.item() in remap and e1.item() in remap)
                             for e0, e1 in zip(data.edge_index[0], data.edge_index[1])]
                ei = (torch.tensor([[remap[e.item()] for e in data.edge_index[:, mask_edge][0]],
                                    [remap[e.item()] for e in data.edge_index[:, mask_edge][1]]], dtype=torch.long)
                      if data.edge_index.size(1) > 0 and any(mask_edge) else torch.zeros((2, 0), dtype=torch.long))
                scores.append(p0 - proba_model(gnn_sum, Data(x=x2, edge_index=ei), cls_idx))
            scores = np.array(scores)
        else:
            torch.manual_seed(SEED + idx)
            batch = torch.zeros(n_nodes, dtype=torch.long, device=device)
            explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
                                    batch=batch, target=torch.tensor([cls_idx], device=device))
            scores = explanation.node_mask.detach().cpu().numpy()
            if scores.ndim > 1:
                scores = scores.sum(axis=1)

        # Classement complet, du score le plus eleve au plus faible
        ranking = np.argsort(-scores)
        ranked_macs = [macs[r] for r in ranking]
        if ref_mac in ranked_macs:
            rang = ranked_macs.index(ref_mac) + 1
        else:
            rang = None

        rows.append({
            'attaque': label,
            'top1': rang == 1,
            'top3': (rang is not None) and (rang <= 3),
            'top5': (rang is not None) and (rang <= 5),
            'reciprocal_rank': (1.0 / rang) if rang is not None else 0.0,
        })

res = pd.DataFrame(rows)
summary = res.groupby('attaque').agg(
    fenetres=('top1', 'size'),
    top1_pourcent=('top1', lambda s: s.mean() * 100),
    top3_pourcent=('top3', lambda s: s.mean() * 100),
    top5_pourcent=('top5', lambda s: s.mean() * 100),
    mrr=('reciprocal_rank', 'mean'),
)
print("\n=== RCA avec Top-1, Top-3, Top-5 et MRR ===")
print(summary.round(3))

summary.to_csv(f'{checkpoint_dir_v4}/diagnostic_rca_topk_mrr_v4.csv')
print("\nResultat sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Determinisme complet active.


/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,115,116,117,118,153,155,157,158,159,160,161,162,163,164,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,247,248,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,86,90,91,92,93,94,96,98,99,100,101,102,103,104,105,107,108,109,110,111,112,113,114,115,116,117,153,155,157,158,159,160,161,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,250,251) have mixed types. Specify dtype option on import or set lo

Deauth: adresses MAC extraites


/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,90,91,92,98,99,100,101,102,103,104,105,107,108,109,110,111,115,119,120,135,137,140,144,146,148,149,151,167,168,178,180,183,184,185,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (71,73,77,78,79,80,81,82,83,84,85,86,90,91,92,93,94,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,122,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,191,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_46324/3030749194.py:35: DtypeW

Disas: adresses MAC extraites


/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,116,117,118,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,118,132,166,167,168,183,184,185,189,190,192,193,194,195,196,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (20,21,43,44,45,62,

... (19 outputs left out)

/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (23,24,48,50,51,81,82,83,84,85,86,90,91,92,93,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (82,84,85,86,87,90,91,92,93,94,96,98,99,100,101,102,103,104,105,106,107,108,109,110,111,115,118,166,167,168,178,180,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,249,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_46324/3030749194.py:35: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,87,90,91,92,93,97,98,99,100,101,102,1

Website_spoofing: adresses MAC extraites
Graphes : 154606 | Test : 30922
GNN (pooling par moyenne) recharge.
GNN a pooling par somme reentraine (deterministe).


/tmp/ipykernel_46324/3030749194.py:246: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_46324/3030749194.py:246: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_46324/3030749194.py:246: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_46324/3030749194.py:246: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_46324/3030749194.py:246: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device),


=== RCA avec Top-1, Top-3, Top-5 et MRR ===
                  fenetres  top1_pourcent  top3_pourcent  top5_pourcent    mrr
attaque                                                                       
Botnet                 125         20.000         72.000         92.800  0.476
Evil_Twin              264         62.879         88.258         95.455  0.771
Krack                  222         26.577         72.072         78.829  0.499
Malware                 85          3.529         32.941         88.235  0.271
SSDP                   288         77.778         99.653        100.000  0.887
SSH                    148         14.189         38.514         74.324  0.364
Website_spoofing       246         32.927         44.715         56.098  0.481

Resultat sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch_geometric
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torch_geometric')

import os, gc, time, pickle, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool
from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.loader import DataLoader as PyGDataLoader
from sklearn.model_selection import train_test_split

base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20
MASTER_SEED = 42  # fixe le decoupage train/test et l'echantillon RCA, identique pour les 5 runs
N_RUNS = 5
RUN_SEEDS = [42, 123, 456, 789, 2024]

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}
mac_cols = ['wlan.sa', 'wlan.da', 'wlan.ta', 'wlan.ra', 'wlan.bssid']
role_cols_map = {'wlan.sa': 0, 'wlan.da': 1, 'wlan.ta': 2, 'wlan.ra': 3, 'wlan.bssid': 4}

mac_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        df_raw = read_csv_safe(os.path.join(folder_path, f)).reset_index(drop=True)
        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw; gc.collect(); continue
        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        cols_present = [c for c in mac_cols if c in df_raw.columns] + ['Label']
        df_mac_slice = df_raw.iloc[start:end][cols_present].reset_index(drop=True)
        file_id = f'{attack_name}__{f}'
        df_mac_slice['source_file'] = file_id
        mac_slices[file_id] = df_mac_slice
        del df_raw; gc.collect()
    print(f'{attack_name}: adresses MAC extraites')

df_mac_combined = pd.concat(mac_slices.values(), ignore_index=True)
del mac_slices
gc.collect()

with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'rb') as f:
    le_gnn = pickle.load(f)['le']

graph_data_list, graph_node_macs, graph_labels = [], [], []
for file_id, group in df_mac_combined.groupby('source_file', sort=False):
    mac_cols_present = [c for c in mac_cols if c in group.columns]
    attack_label = file_id.split('__')[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window_df = group.iloc[s:s + window_size]
        edges = set()
        role_counts = {}
        for _, row in window_df.iterrows():
            present = [(c, row[c]) for c in mac_cols_present if pd.notna(row[c])]
            addrs = [a for _, a in present]
            for col, mac in present:
                if mac not in role_counts:
                    role_counts[mac] = [0, 0, 0, 0, 0]
                role_counts[mac][role_cols_map[col]] += 1
            if len(addrs) >= 2:
                edges.add(tuple(sorted(addrs[:2])))
        node_list = sorted(role_counts.keys()) if role_counts else ['isole']
        if 'isole' in node_list:
            role_counts['isole'] = [0, 0, 0, 0, 0]
        node_idx = {mac: idx for idx, mac in enumerate(node_list)}
        edge_index_list = []
        for a, b in edges:
            edge_index_list.append([node_idx[a], node_idx[b]])
            edge_index_list.append([node_idx[b], node_idx[a]])
        x = torch.tensor([role_counts[mac] for mac in node_list], dtype=torch.float)
        edge_index = (torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
                      if edge_index_list else torch.zeros((2, 0), dtype=torch.long))
        window_labels = window_df['Label'].values
        label_name = attack_label if (window_labels != 'Normal').any() else 'Normal'
        graph_data_list.append(Data(x=x, edge_index=edge_index))
        graph_node_macs.append(node_list)
        graph_labels.append(label_name)
del df_mac_combined
gc.collect()

y_gnn_encoded = le_gnn.transform(graph_labels)
for data, y_val in zip(graph_data_list, y_gnn_encoded):
    data.y = torch.tensor([y_val])
graph_data_list = [d.cpu() for d in graph_data_list]

# Decoupage train/test FIXE (ne depend pas du seed de run, pour isoler l'effet de l'entrainement seul)
graph_indices = np.arange(len(graph_data_list))
train_idx_g, test_idx_g = train_test_split(graph_indices, test_size=0.2, random_state=MASTER_SEED, stratify=y_gnn_encoded)
print("Graphes :", len(graph_data_list), "| Test :", len(test_idx_g))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GCN_Mean(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GCN_Sum(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_add_pool(x, batch)
        return self.lin(x)

def proba_model(model, data, cls_idx):
    with torch.no_grad():
        d = data.to(device)
        batch = torch.zeros(d.x.size(0), dtype=torch.long, device=device)
        out = model(d.x, d.edge_index, batch)
        return F.softmax(out, dim=1)[0, cls_idx].item()

reference = {
    'Malware': '00:0c:29:cf:08:aa', 'SSH': '00:0c:29:1d:70:f5',
    'Botnet': '04:ed:33:e0:24:82', 'Krack': '94:e9:79:82:c5:77',
    'Website_spoofing': '04:ed:33:e0:24:82', 'SSDP': '04:ed:33:e0:24:82',
}
reference_sum = {'Evil_Twin': '0c:9d:92:54:fe:35'}

# Echantillon RCA FIXE (meme fenetres testees a chaque run, pour une comparaison equitable)
rng_master = np.random.RandomState(MASTER_SEED)
samples_by_label = {}
for label in list(reference.keys()) + list(reference_sum.keys()):
    pool = [i for i in test_idx_g if graph_labels[i] == label]
    samples_by_label[label] = rng_master.choice(pool, size=min(300, len(pool)), replace=False)

all_runs_results = []
train_graphs = [graph_data_list[i] for i in train_idx_g]

for run_num, seed in enumerate(RUN_SEEDS):
    print(f"\n{'='*60}\nRUN {run_num + 1}/{N_RUNS} (seed={seed})\n{'='*60}")
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    gnn_mean = GCN_Mean(num_features=5, num_classes=len(le_gnn.classes_)).to(device)
    optimizer = torch.optim.Adam(gnn_mean.parameters(), lr=0.01)
    train_loader = PyGDataLoader(train_graphs, batch_size=128, shuffle=True)
    for epoch in range(15):
        gnn_mean.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = gnn_mean(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)
            loss.backward()
            optimizer.step()
    print("GNN moyenne entraine.")

    gnn_sum = GCN_Sum(num_features=5, num_classes=len(le_gnn.classes_)).to(device)
    optimizer2 = torch.optim.Adam(gnn_sum.parameters(), lr=0.01)
    for epoch in range(15):
        gnn_sum.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer2.zero_grad()
            out = gnn_sum(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)
            loss.backward()
            optimizer2.step()
    print("GNN somme entraine.")

    explainer = Explainer(
        model=gnn_mean, algorithm=GNNExplainer(epochs=100),
        explanation_type='model', node_mask_type='object', edge_mask_type=None,
        model_config=dict(mode='multiclass_classification', task_level='graph', return_type='raw'),
    )

    for label in list(reference.keys()) + list(reference_sum.keys()):
        ref_mac = reference.get(label) or reference_sum.get(label)
        model_use = gnn_sum if label in reference_sum else gnn_mean
        cls_idx = int(le_gnn.transform([label])[0])

        for idx in samples_by_label[label]:
            data = graph_data_list[idx]
            macs = graph_node_macs[idx]
            n_nodes = data.x.size(0)
            if n_nodes < 2:
                continue
            all_proba = [proba_model(model_use, data, c) for c in range(len(le_gnn.classes_))]
            pred0 = le_gnn.inverse_transform([int(np.argmax(all_proba))])[0]
            if pred0 != label:
                continue

            if label in reference_sum:
                p0 = all_proba[cls_idx]
                scores = []
                for node in range(n_nodes):
                    keep = [i for i in range(n_nodes) if i != node]
                    remap = {old: new for new, old in enumerate(keep)}
                    x2 = data.x[keep]
                    mask_edge = [(e0.item() in remap and e1.item() in remap)
                                 for e0, e1 in zip(data.edge_index[0], data.edge_index[1])]
                    ei = (torch.tensor([[remap[e.item()] for e in data.edge_index[:, mask_edge][0]],
                                        [remap[e.item()] for e in data.edge_index[:, mask_edge][1]]], dtype=torch.long)
                          if data.edge_index.size(1) > 0 and any(mask_edge) else torch.zeros((2, 0), dtype=torch.long))
                    scores.append(p0 - proba_model(gnn_sum, Data(x=x2, edge_index=ei), cls_idx))
                scores = np.array(scores)
            else:
                torch.manual_seed(seed * 1000 + idx)
                batch = torch.zeros(n_nodes, dtype=torch.long, device=device)
                explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
                                        batch=batch, target=torch.tensor([cls_idx], device=device))
                scores = explanation.node_mask.detach().cpu().numpy()
                if scores.ndim > 1:
                    scores = scores.sum(axis=1)

            ranking = np.argsort(-scores)
            ranked_macs = [macs[r] for r in ranking]
            rang = ranked_macs.index(ref_mac) + 1 if ref_mac in ranked_macs else None

            all_runs_results.append({
                'run': run_num, 'attaque': label,
                'top1': rang == 1,
                'top3': (rang is not None) and (rang <= 3),
                'top5': (rang is not None) and (rang <= 5),
                'reciprocal_rank': (1.0 / rang) if rang is not None else 0.0,
            })

    del gnn_mean, gnn_sum, explainer
    gc.collect()
    torch.cuda.empty_cache()

res_all = pd.DataFrame(all_runs_results)

per_run = res_all.groupby(['attaque', 'run']).agg(
    top1=('top1', 'mean'), top3=('top3', 'mean'), top5=('top5', 'mean'), mrr=('reciprocal_rank', 'mean')
).reset_index()

summary = per_run.groupby('attaque').agg(
    top1_moyenne=('top1', lambda s: s.mean() * 100), top1_ecart_type=('top1', lambda s: s.std() * 100),
    top3_moyenne=('top3', lambda s: s.mean() * 100), top3_ecart_type=('top3', lambda s: s.std() * 100),
    top5_moyenne=('top5', lambda s: s.mean() * 100), top5_ecart_type=('top5', lambda s: s.std() * 100),
    mrr_moyenne=('mrr', 'mean'), mrr_ecart_type=('mrr', 'std'),
)
print("\n=== RCA, moyenne et ecart-type sur 5 entrainements independants ===")
print(summary.round(3))

per_run.to_csv(f'{checkpoint_dir_v4}/rca_topk_mrr_par_run_v4.csv', index=False)
summary.to_csv(f'{checkpoint_dir_v4}/rca_topk_mrr_moyenne_v4.csv')
print("\nResultats sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_18249/4263604150.py:29: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,115,116,117,118,153,155,157,158,159,160,161,162,163,164,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,247,248,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_18249/4263604150.py:29: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,86,90,91,92,93,94,96,98,99,100,101,102,103,104,105,107,108,109,110,111,112,113,114,115,116,117,153,155,157,158,159,160,161,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,250,251) have mixed types. Specify dtype option on import or set lo

Deauth: adresses MAC extraites


/tmp/ipykernel_18249/4263604150.py:29: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,86,90,91,92,98,99,100,101,102,103,104,105,107,108,109,110,111,115,119,120,135,137,140,144,146,148,149,151,167,168,178,180,183,184,185,190,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_18249/4263604150.py:29: DtypeWarning: Columns (71,73,77,78,79,80,81,82,83,84,85,86,90,91,92,93,94,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,122,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,191,192,193,194,195,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,244,247,248,250,251,252) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_18249/4263604150.py:29: DtypeW

Disas: adresses MAC extraites


/tmp/ipykernel_18249/4263604150.py:29: DtypeWarning: Columns (20,21,43,44,45,62,65,71,73,77,78,79,80,82,84,85,87,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,112,113,114,116,117,118,166,167,168,169,170,171,172,173,175,176,177,178,180,181,183,184,185,189,190,192,193,194,195,196,197,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_18249/4263604150.py:29: DtypeWarning: Columns (71,73,77,78,79,80,82,84,85,90,91,92,93,94,96,98,99,100,101,102,103,104,107,108,109,110,111,118,132,166,167,168,183,184,185,189,190,192,193,194,195,196,200,201,202,203,204,205,206,209,210,211,212,213,215,217,218,219,247,248,249,250,251) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, usecols=usecols)
/tmp/ipykernel_18249/4263604150.py:29: DtypeWarning: Columns (20,21,43,44,45,62,

... (27 outputs left out)

/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device),


RUN 5/5 (seed=2024)
GNN moyenne entraine.
GNN somme entraine.


/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
/tmp/ipykernel_18249/4263604150.py:253: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(x=data.x.to(device),


=== RCA, moyenne et ecart-type sur 5 entrainements independants ===
                  top1_moyenne  top1_ecart_type  top3_moyenne  \
attaque                                                         
Botnet                  14.869            4.727        60.315   
Evil_Twin               57.283           13.467        84.836   
Krack                   32.129           12.678        69.444   
Malware                  4.386            4.632        40.032   
SSDP                    20.306           36.976        96.468   
SSH                      1.619            2.529        11.266   
Website_spoofing        37.576            4.130        57.517   

                  top3_ecart_type  top5_moyenne  top5_ecart_type  mrr_moyenne  \
attaque                                                                         
Botnet                      6.866        94.921            1.734        0.422   
Evil_Twin                   9.089        97.039            2.629        0.730   
Krack                

In [ ]:
import warnings
warnings.simplefilter('ignore')
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch_geometric

import os, gc, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool
from torch_geometric.explain import Explainer, GNNExplainer
from sklearn.model_selection import train_test_split

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
window_size = 20

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}
mac_cols = ['wlan.sa', 'wlan.da', 'wlan.ta', 'wlan.ra', 'wlan.bssid']
role_cols_map = {'wlan.sa': 0, 'wlan.da': 1, 'wlan.ta': 2, 'wlan.ra': 3, 'wlan.bssid': 4}
wifi_domain = {'Deauth', 'Disas', 'Re-Assoc', 'Rogue_AP', 'Krack', 'Kr00k', 'Evil_Twin'}
reseau_domain = {'SSH', 'Botnet', 'Malware', 'SQL_Injection', 'SSDP', 'Website_spoofing'}
reference = {
    'Malware': '00:0c:29:cf:08:aa', 'SSH': '00:0c:29:1d:70:f5',
    'Botnet': '04:ed:33:e0:24:82', 'Krack': '94:e9:79:82:c5:77',
    'Website_spoofing': '04:ed:33:e0:24:82', 'SSDP': '04:ed:33:e0:24:82',
}
reference_sum = {'Evil_Twin': '0c:9d:92:54:fe:35'}

mac_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        df_raw = read_csv_safe(os.path.join(folder_path, f)).reset_index(drop=True)
        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw; gc.collect(); continue
        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        cols_present = [c for c in mac_cols if c in df_raw.columns] + ['Label']
        df_mac_slice = df_raw.iloc[start:end][cols_present].reset_index(drop=True)
        file_id = f'{attack_name}__{f}'
        df_mac_slice['source_file'] = file_id
        mac_slices[file_id] = df_mac_slice
        del df_raw; gc.collect()
    print(f'{attack_name}: adresses MAC extraites')

df_mac_combined = pd.concat(mac_slices.values(), ignore_index=True)
del mac_slices
gc.collect()

with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'rb') as f:
    le_gnn = pickle.load(f)['le']

graph_data_list, graph_node_macs, graph_labels, graph_file = [], [], [], []
for file_id, group in df_mac_combined.groupby('source_file', sort=False):
    mac_cols_present = [c for c in mac_cols if c in group.columns]
    attack_label = file_id.split('__')[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window_df = group.iloc[s:s + window_size]
        edges = set()
        role_counts = {}
        for _, row in window_df.iterrows():
            present = [(c, row[c]) for c in mac_cols_present if pd.notna(row[c])]
            addrs = [a for _, a in present]
            for col, mac in present:
                if mac not in role_counts:
                    role_counts[mac] = [0, 0, 0, 0, 0]
                role_counts[mac][role_cols_map[col]] += 1
            if len(addrs) >= 2:
                edges.add(tuple(sorted(addrs[:2])))
        node_list = sorted(role_counts.keys()) if role_counts else ['isole']
        if 'isole' in node_list:
            role_counts['isole'] = [0, 0, 0, 0, 0]
        node_idx = {mac: idx for idx, mac in enumerate(node_list)}
        edge_index_list = []
        for a, b in edges:
            edge_index_list.append([node_idx[a], node_idx[b]])
            edge_index_list.append([node_idx[b], node_idx[a]])
        x = torch.tensor([role_counts[mac] for mac in node_list], dtype=torch.float)
        edge_index = (torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
                      if edge_index_list else torch.zeros((2, 0), dtype=torch.long))
        window_labels = window_df['Label'].values
        label_name = attack_label if (window_labels != 'Normal').any() else 'Normal'
        graph_data_list.append(Data(x=x, edge_index=edge_index))
        graph_node_macs.append(node_list)
        graph_labels.append(label_name)
        graph_file.append(file_id)
del df_mac_combined
gc.collect()
print("Graphes reconstruits :", len(graph_data_list))

y_gnn_encoded = le_gnn.transform(graph_labels)
for data, y_val in zip(graph_data_list, y_gnn_encoded):
    data.y = torch.tensor([y_val])
graph_data_list = [d.cpu() for d in graph_data_list]

graph_indices = np.arange(len(graph_data_list))
train_idx_g, test_idx_g = train_test_split(graph_indices, test_size=0.2, random_state=42, stratify=y_gnn_encoded)
test_idx_set = set(test_idx_g)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GCN_Mean(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index)); x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch); return self.lin(x)

class GCN_Sum(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index)); x = F.relu(self.conv2(x, edge_index))
        x = global_add_pool(x, batch); return self.lin(x)

gnn_mean = GCN_Mean(5, len(le_gnn.classes_)).to(device)
gnn_mean.load_state_dict(torch.load(f'{checkpoint_dir_v4}/gnn_v4.pt', map_location=device))
gnn_mean.eval()
gnn_sum = GCN_Sum(5, len(le_gnn.classes_)).to(device)
gnn_sum.load_state_dict(torch.load(f'{checkpoint_dir_v4}/gnn_sum_v4.pt', map_location=device))
gnn_sum.eval()

def proba_model(model, data, cls_idx):
    with torch.no_grad():
        d = data.to(device)
        batch = torch.zeros(d.x.size(0), dtype=torch.long, device=device)
        out = model(d.x, d.edge_index, batch)
        return F.softmax(out, dim=1)[0, cls_idx].item()

explainer = Explainer(
    model=gnn_mean, algorithm=GNNExplainer(epochs=100),
    explanation_type='model', node_mask_type='object', edge_mask_type=None,
    model_config=dict(mode='multiclass_classification', task_level='graph', return_type='raw'),
)

# RCA sur TOUTES les fenetres de test des attaques concernees (pas un echantillon de 300)
rca_rows = []
all_labels_rca = list(reference.keys()) + list(reference_sum.keys())
for idx in test_idx_g:
    label = graph_labels[idx]
    if label not in all_labels_rca:
        continue
    data = graph_data_list[idx]
    macs = graph_node_macs[idx]
    n_nodes = data.x.size(0)
    if n_nodes < 2:
        continue
    ref_mac = reference.get(label) or reference_sum.get(label)
    model_use = gnn_sum if label in reference_sum else gnn_mean
    cls_idx = int(le_gnn.transform([label])[0])
    all_proba = [proba_model(model_use, data, c) for c in range(len(le_gnn.classes_))]
    pred0 = le_gnn.inverse_transform([int(np.argmax(all_proba))])[0]
    if pred0 != label:
        continue
    if label in reference_sum:
        p0 = all_proba[cls_idx]
        scores = []
        for node in range(n_nodes):
            keep = [i for i in range(n_nodes) if i != node]
            remap = {old: new for new, old in enumerate(keep)}
            x2 = data.x[keep]
            mask_edge = [(e0.item() in remap and e1.item() in remap)
                         for e0, e1 in zip(data.edge_index[0], data.edge_index[1])]
            ei = (torch.tensor([[remap[e.item()] for e in data.edge_index[:, mask_edge][0]],
                                [remap[e.item()] for e in data.edge_index[:, mask_edge][1]]], dtype=torch.long)
                  if data.edge_index.size(1) > 0 and any(mask_edge) else torch.zeros((2, 0), dtype=torch.long))
            scores.append(p0 - proba_model(gnn_sum, Data(x=x2, edge_index=ei), cls_idx))
        scores = np.array(scores)
    else:
        batch = torch.zeros(n_nodes, dtype=torch.long, device=device)
        explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
                                batch=batch, target=torch.tensor([cls_idx], device=device))
        scores = explanation.node_mask.detach().cpu().numpy()
        if scores.ndim > 1:
            scores = scores.sum(axis=1)
    device_designe = macs[int(np.argmax(scores))]
    rca_rows.append({
        'graph_idx': idx, 'source_file': graph_file[idx], 'attaque': label,
        'domaine': 'Wi-Fi' if label in wifi_domain else 'Reseau',
        'appareil_designe': device_designe, 'appareil_correct': device_designe == ref_mac,
    })
    if len(rca_rows) % 500 == 0:
        print(f"  {len(rca_rows)} fenetres RCA traitees...")

rca_df = pd.DataFrame(rca_rows)
rca_df.to_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')
print(f"\nRCA complet sur {len(rca_df)} fenetres de test, precision globale : {rca_df['appareil_correct'].mean():.4f}")
print("Fichier sauvegarde.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.4 MB/s eta 0:00:00
Deauth: adresses MAC extraites
Disas: adresses MAC extraites
Re-Assoc: adresses MAC extraites
Rogue_AP: adresses MAC extraites
Krack: adresses MAC extraites
Kr00k: adresses MAC extraites
SSH: adresses MAC extraites
Botnet: adresses MAC extraites
Malware: adresses MAC extraites
SQL_Injection: adresses MAC extraites
SSDP: adresses MAC extraites
Evil_Twin: adresses MAC extraites
Website_spoofing: adresses MAC extraites
Graphes reconstruits : 154606
  500 fenetres RCA traitees...
  1000 fenetres RCA traitees...
  1500 fenetres RCA traitees...
  2000 fenetres RCA traitees...
  2500 fenetres RCA traitees...
  3000 fenetres RCA traitees...
  3500 fenetres RCA traitees...
  4000 fenetres RCA traitees...
  4500 fenetres RCA traitees...
  5000 fenetres RCA traitees...
  5500 fenetres RCA traitees...
  6000 fenetres 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']
feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')

rows = []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        wl = labels[s:s + window_size]
        label_name = attack_name if (wl != 'Normal').any() else 'Normal'
        rows.append({'source_file': file_id, 'position': i, 'vraie_etiquette': label_name,
                     'attaque_fichier': attack_name})
df_windows_meta = pd.DataFrame(rows)
del df_encoded
gc.collect()
print("Fenetres totales :", len(df_windows_meta))

data_hybrid = np.load(f'{checkpoint_dir_v4}/hybrid_proba_v4.npz')
hybrid_proba = 0.4 * data_hybrid['xgb_proba'] + 0.4 * data_hybrid['lstm_proba'] + 0.2 * data_hybrid['gnn_proba']
y_test_aligned = data_hybrid['y_test']

all_idx = np.arange(len(df_windows_meta))
y_full_encoded = le_max.transform(df_windows_meta['vraie_etiquette'].values)
_, test_idx = train_test_split(all_idx, test_size=0.2, random_state=42, stratify=y_full_encoded)

df_test = df_windows_meta.iloc[test_idx].reset_index(drop=True)
df_test['prediction_hybride'] = le_max.inverse_transform(hybrid_proba.argmax(axis=1))
df_test['confiance_hybride'] = hybrid_proba.max(axis=1)

rca_df = pd.read_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')
rca_small = rca_df[['source_file', 'appareil_designe', 'appareil_correct']].copy()
rca_small['position'] = rca_df.groupby('source_file').cumcount()

wifi_domain = {'Deauth', 'Disas', 'Re-Assoc', 'Rogue_AP', 'Krack', 'Kr00k', 'Evil_Twin'}

env_df = df_test.copy()
env_df['domaine'] = env_df['attaque_fichier'].apply(lambda a: 'Wi-Fi' if a in wifi_domain else 'Reseau')
env_df = env_df.sort_values(['source_file', 'position']).reset_index(drop=True)

appareil_recent = []
repetition_count = []
historique_par_fichier = {}
for _, row in env_df.iterrows():
    fid = row['source_file']
    hist = historique_par_fichier.setdefault(fid, [])
    match = rca_df[(rca_df['source_file'] == fid)]
    appareil_recent.append(None)
    repetition_count.append(0)

env_df.to_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_v4.parquet')
rca_df.to_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')
print("\nEnvironnement de base construit :", env_df.shape)
print(env_df['domaine'].value_counts())
print("\nFichier sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fenetres totales : 154606

Environnement de base construit : (30922, 7)
domaine
Wi-Fi     16547
Reseau    14375
Name: count, dtype: int64

Fichier sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']
feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
rows = []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        wl = labels[s:s + window_size]
        label_name = attack_name if (wl != 'Normal').any() else 'Normal'
        rows.append({'source_file': file_id, 'position': i, 'vraie_etiquette': label_name,
                     'attaque_fichier': attack_name})
df_windows_meta = pd.DataFrame(rows)
del df_encoded
gc.collect()
print("Fenetres totales :", len(df_windows_meta))

# Decoupage par FICHIERS entiers, pas par fenetres
fichiers_uniques = df_windows_meta[['source_file', 'attaque_fichier']].drop_duplicates()
fichiers_train, fichiers_test = train_test_split(
    fichiers_uniques['source_file'].values, test_size=0.2, random_state=42,
    stratify=fichiers_uniques['attaque_fichier'].values
)
print(f"Fichiers entraines : {len(fichiers_train)} | Fichiers test : {len(fichiers_test)}")

df_test = df_windows_meta[df_windows_meta['source_file'].isin(fichiers_test)].reset_index(drop=True)
df_train_files = df_windows_meta[df_windows_meta['source_file'].isin(fichiers_train)]
print(f"Fenetres d'entrainement (hors environnement) : {len(df_train_files)} | Fenetres de test (environnement) : {len(df_test)}")

# Recalcul des probabilites hybrides UNIQUEMENT pour les fenetres de ces fichiers de test
xgb_combo = pickle.load(open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb'))
import joblib
xgb_model = joblib.load(f'{checkpoint_dir_v4}/xgb_combo_v4.joblib')

from tensorflow.keras import models as keras_models
lstm_model = keras_models.load_model(f'{checkpoint_dir_v4}/lstm_v4.keras')
with open(f'{checkpoint_dir_v4}/scaler_lstm_v4.pkl', 'rb') as f:
    scaler_lstm = pickle.load(f)['scaler']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
df_encoded_test = df_encoded[df_encoded['source_file'].isin(fichiers_test)]

X_max_test, X_seq_test = [], []
for file_id, group in df_encoded_test.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window = features[s:s + window_size]
        X_max_test.append(window.max(axis=0))
        X_seq_test.append(window)
del df_encoded, df_encoded_test
gc.collect()
X_max_test = np.array(X_max_test, dtype=np.float32)
X_seq_test = np.array(X_seq_test, dtype=np.float32)

xgb_proba_env = xgb_model.predict_proba(X_max_test)
n_t, n_f = X_seq_test.shape[1], X_seq_test.shape[2]
X_seq_scaled = scaler_lstm.transform(X_seq_test.reshape(-1, n_f)).astype(np.float32).reshape(X_seq_test.shape)
lstm_proba_env = lstm_model.predict(X_seq_scaled, batch_size=256)

print("\nVerification alignement :", len(df_test), "==", len(xgb_proba_env), "==", len(lstm_proba_env))

df_test['confiance_xgb_lstm'] = (0.5 * xgb_proba_env + 0.5 * lstm_proba_env).max(axis=1)
df_test['prediction_xgb_lstm'] = le_max.inverse_transform((0.5 * xgb_proba_env + 0.5 * lstm_proba_env).argmax(axis=1))

wifi_domain = {'Deauth', 'Disas', 'Re-Assoc', 'Rogue_AP', 'Krack', 'Kr00k', 'Evil_Twin'}
df_test['domaine'] = df_test['attaque_fichier'].apply(lambda a: 'Wi-Fi' if a in wifi_domain else 'Reseau')
df_test = df_test.sort_values(['source_file', 'position']).reset_index(drop=True)

df_test.to_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_v4.parquet')
pd.Series(fichiers_train).to_csv(f'{checkpoint_dir_v4}/fichiers_train_auto_guerison_v4.csv', index=False)
pd.Series(fichiers_test).to_csv(f'{checkpoint_dir_v4}/fichiers_test_auto_guerison_v4.csv', index=False)

print("\nEnvironnement de base reconstruit, par fichiers :", df_test.shape)
print(df_test['domaine'].value_counts())
print("\nFichier sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fenetres totales : 154606
Fichiers entraines : 51 | Fichiers test : 13
Fenetres d'entrainement (hors environnement) : 124526 | Fenetres de test (environnement) : 30080
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Verification alignement : 30080 == 30080 == 30080

Environnement de base reconstruit, par fichiers : (30080, 7)
domaine
Wi-Fi     16249
Reseau    13831
Name: count, dtype: int64

Fichier sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch_geometric
import warnings
warnings.simplefilter('ignore')

import os, gc, time, pickle, joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import train_test_split

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
window_size = 20

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']
feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
rows = []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    labels = group['Label'].values
    attack_name = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        wl = labels[s:s + window_size]
        label_name = attack_name if (wl != 'Normal').any() else 'Normal'
        rows.append({'source_file': file_id, 'position': i, 'vraie_etiquette': label_name,
                     'attaque_fichier': attack_name})
df_windows_meta = pd.DataFrame(rows)
print("Fenetres totales :", len(df_windows_meta))

fichiers_uniques = df_windows_meta[['source_file', 'attaque_fichier']].drop_duplicates()
fichiers_train, fichiers_test = train_test_split(
    fichiers_uniques['source_file'].values, test_size=0.2, random_state=42,
    stratify=fichiers_uniques['attaque_fichier'].values
)
print(f"Fichiers entraines : {len(fichiers_train)} | Fichiers test : {len(fichiers_test)}")

df_test = df_windows_meta[df_windows_meta['source_file'].isin(fichiers_test)].reset_index(drop=True)
print(f"Fenetres de test (environnement) : {len(df_test)}")

# --- XGBoost + LSTM, sur les fenetres des fichiers de test ---
xgb_model = joblib.load(f'{checkpoint_dir_v4}/xgb_combo_v4.joblib')
from tensorflow.keras import models as keras_models
lstm_model = keras_models.load_model(f'{checkpoint_dir_v4}/lstm_v4.keras')
with open(f'{checkpoint_dir_v4}/scaler_lstm_v4.pkl', 'rb') as f:
    scaler_lstm = pickle.load(f)['scaler']

df_encoded_test = df_encoded[df_encoded['source_file'].isin(fichiers_test)]
X_max_test, X_seq_test = [], []
for file_id, group in df_encoded_test.groupby('source_file', sort=False):
    features = group[feature_cols].values.astype(np.float32)
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window = features[s:s + window_size]
        X_max_test.append(window.max(axis=0))
        X_seq_test.append(window)
del df_encoded, df_encoded_test
gc.collect()
X_max_test = np.array(X_max_test, dtype=np.float32)
X_seq_test = np.array(X_seq_test, dtype=np.float32)

xgb_proba_env = xgb_model.predict_proba(X_max_test)
n_t, n_f = X_seq_test.shape[1], X_seq_test.shape[2]
X_seq_scaled = scaler_lstm.transform(X_seq_test.reshape(-1, n_f)).astype(np.float32).reshape(X_seq_test.shape)
lstm_proba_env = lstm_model.predict(X_seq_scaled, batch_size=256)
print("XGBoost et LSTM recalcules :", xgb_proba_env.shape, lstm_proba_env.shape)
del X_max_test, X_seq_test
gc.collect()

# --- GNN, reconstruction des graphes pour les memes fichiers de test ---
folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}
mac_cols = ['wlan.sa', 'wlan.da', 'wlan.ta', 'wlan.ra', 'wlan.bssid']
role_cols_map = {'wlan.sa': 0, 'wlan.da': 1, 'wlan.ta': 2, 'wlan.ra': 3, 'wlan.bssid': 4}

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

fichiers_test_set = set(fichiers_test)
mac_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        file_id = f'{attack_name}__{f}'
        if file_id not in fichiers_test_set:
            continue
        df_raw = read_csv_safe(os.path.join(folder_path, f)).reset_index(drop=True)
        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw; gc.collect(); continue
        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        cols_present = [c for c in mac_cols if c in df_raw.columns] + ['Label']
        df_mac_slice = df_raw.iloc[start:end][cols_present].reset_index(drop=True)
        df_mac_slice['source_file'] = file_id
        mac_slices[file_id] = df_mac_slice
        del df_raw; gc.collect()

df_mac_combined = pd.concat(mac_slices.values(), ignore_index=True)
del mac_slices
gc.collect()

with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'rb') as f:
    le_gnn = pickle.load(f)['le']

graph_data_list = []
for file_id, group in df_mac_combined.groupby('source_file', sort=False):
    mac_cols_present = [c for c in mac_cols if c in group.columns]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window_df = group.iloc[s:s + window_size]
        edges = set()
        role_counts = {}
        for _, row in window_df.iterrows():
            present = [(c, row[c]) for c in mac_cols_present if pd.notna(row[c])]
            addrs = [a for _, a in present]
            for col, mac in present:
                if mac not in role_counts:
                    role_counts[mac] = [0, 0, 0, 0, 0]
                role_counts[mac][role_cols_map[col]] += 1
            if len(addrs) >= 2:
                edges.add(tuple(sorted(addrs[:2])))
        node_list = sorted(role_counts.keys()) if role_counts else ['isole']
        if 'isole' in node_list:
            role_counts['isole'] = [0, 0, 0, 0, 0]
        node_idx = {mac: idx for idx, mac in enumerate(node_list)}
        edge_index_list = []
        for a, b in edges:
            edge_index_list.append([node_idx[a], node_idx[b]])
            edge_index_list.append([node_idx[b], node_idx[a]])
        x = torch.tensor([role_counts[mac] for mac in node_list], dtype=torch.float)
        edge_index = (torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
                      if edge_index_list else torch.zeros((2, 0), dtype=torch.long))
        graph_data_list.append(Data(x=x, edge_index=edge_index))
del df_mac_combined
gc.collect()
print("Graphes reconstruits (fichiers de test) :", len(graph_data_list))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class GCN_Mean(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index)); x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch); return self.lin(x)

gnn_model = GCN_Mean(5, len(le_gnn.classes_)).to(device)
gnn_model.load_state_dict(torch.load(f'{checkpoint_dir_v4}/gnn_v4.pt', map_location=device))
gnn_model.eval()

gnn_proba_env = []
with torch.no_grad():
    for g in graph_data_list:
        g = g.to(device)
        batch = torch.zeros(g.x.size(0), dtype=torch.long, device=device)
        out = gnn_model(g.x, g.edge_index, batch)
        gnn_proba_env.append(F.softmax(out, dim=1).cpu().numpy()[0])
gnn_proba_env = np.array(gnn_proba_env)
print("GNN calcule :", gnn_proba_env.shape)
print("Verification alignement final :", len(df_test), "==", xgb_proba_env.shape[0], "==", lstm_proba_env.shape[0], "==", gnn_proba_env.shape[0])

# --- Hybride complet, 0.4/0.4/0.2 ---
hybrid_proba_env = 0.4 * xgb_proba_env + 0.4 * lstm_proba_env + 0.2 * gnn_proba_env
df_test['confiance_hybride'] = hybrid_proba_env.max(axis=1)
df_test['prediction_hybride'] = le_max.inverse_transform(hybrid_proba_env.argmax(axis=1))

wifi_domain = {'Deauth', 'Disas', 'Re-Assoc', 'Rogue_AP', 'Krack', 'Kr00k', 'Evil_Twin'}
df_test['domaine'] = df_test['attaque_fichier'].apply(lambda a: 'Wi-Fi' if a in wifi_domain else 'Reseau')
df_test = df_test.sort_values(['source_file', 'position']).reset_index(drop=True)

df_test.to_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_v4.parquet')
pd.Series(fichiers_train).to_csv(f'{checkpoint_dir_v4}/fichiers_train_auto_guerison_v4.csv', index=False)
pd.Series(fichiers_test).to_csv(f'{checkpoint_dir_v4}/fichiers_test_auto_guerison_v4.csv', index=False)

print("\nEnvironnement corrige (Hybride complet, 3 modeles), par fichiers :", df_test.shape)
print(df_test['domaine'].value_counts())
print("\nFichier sauvegarde, remplace la version precedente.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fenetres totales : 154606
Fichiers entraines : 51 | Fichiers test : 13
Fenetres de test (environnement) : 30080
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
XGBoost et LSTM recalcules : (30080, 14) (30080, 14)
Graphes reconstruits (fichiers de test) : 30080
GNN calcule : (30080, 14)
Verification alignement final : 30080 == 30080 == 30080 == 30080

Environnement corrige (Hybride complet, 3 modeles), par fichiers : (30080, 7)
domaine
Wi-Fi     16249
Reseau    13831
Name: count, dtype: int64

Fichier sauvegarde, remplace la version precedente.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
fenetre_memoire = 200

env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_v4.parquet')
rca_df = pd.read_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')

rca_df = rca_df.sort_values(['source_file']).reset_index(drop=True)
rca_df['position'] = rca_df.groupby('source_file').cumcount()
rca_lookup = rca_df.set_index(['source_file', 'position'])['appareil_designe'].to_dict()

env_df = env_df.sort_values(['source_file', 'position']).reset_index(drop=True)

incidents_recents = []
repetition_appareil = []
appareil_present_calcule = []

for file_id, group in env_df.groupby('source_file', sort=False):
    positions = group['position'].values
    historique_incidents = []
    historique_appareils = []
    for pos in positions:
        fenetre_debut = pos - fenetre_memoire
        while historique_incidents and historique_incidents[0] < fenetre_debut:
            historique_incidents.pop(0)
            historique_appareils.pop(0)

        incidents_recents.append(len(historique_incidents))
        appareil_ici = rca_lookup.get((file_id, pos), None)
        appareil_present_calcule.append(appareil_ici)
        if appareil_ici is not None and len(historique_appareils) > 0:
            repetition_appareil.append(historique_appareils.count(appareil_ici))
        else:
            repetition_appareil.append(0)

        row_mask = (group['position'] == pos)
        label_ici = group.loc[row_mask, 'vraie_etiquette'].iloc[0]
        if label_ici != 'Normal':
            historique_incidents.append(pos)
            historique_appareils.append(appareil_ici)

env_df['incidents_recents_200'] = incidents_recents
env_df['repetition_appareil'] = repetition_appareil
env_df['appareil_designe'] = appareil_present_calcule

print("Fenetres avec RCA calcule :", env_df['appareil_designe'].notna().sum())
print("\nDistribution des incidents recents :")
print(env_df['incidents_recents_200'].describe())
print("\nDistribution de la repetition d'appareil :")
print(env_df['repetition_appareil'].describe())

env_df.to_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')
print("\nEtat MAPE-K sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fenetres avec RCA calcule : 1346

Distribution des incidents recents :
count    30080.000000
mean        96.246775
std         74.405224
min          0.000000
25%         23.000000
50%         80.000000
75%        177.000000
max        200.000000
Name: incidents_recents_200, dtype: float64

Distribution de la repetition d'appareil :
count    30080.000000
mean         2.017055
std         15.721643
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        169.000000
Name: repetition_appareil, dtype: float64

Etat MAPE-K sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch_geometric
import warnings
warnings.simplefilter('ignore')

import os, gc, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool
from torch_geometric.explain import Explainer, GNNExplainer

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
base_path = '/content/drive/MyDrive/EXTRAITFULL/CSV'
window_size = 20

fichiers_test = pd.read_csv(f'{checkpoint_dir_v4}/fichiers_test_auto_guerison_v4.csv')['0'].tolist()
fichiers_test_set = set(fichiers_test)
print("Fichiers de test (environnement) :", len(fichiers_test_set))

folder_names = {
    'Deauth': '1.Deauth', 'Disas': '2.Disas', 'Re-Assoc': '3.(Re)Assoc',
    'Rogue_AP': '4.Rogue_AP', 'Krack': '5.Krack', 'Kr00k': '6.Kr00k',
    'SSH': '7.SSH', 'Botnet': '8.Botnet', 'Malware': '9.Malware',
    'SQL_Injection': '10.SQL_Injection', 'SSDP': '11.SSDP',
    'Evil_Twin': '12.Evil_Twin', 'Website_spoofing': '13.Website_spoofing',
}
known_files_5 = {
    'Deauth': ['Deauth_30.csv', 'Deauth_22.csv', 'Deauth_31.csv', 'Deauth_23.csv', 'Deauth_29.csv'],
    'Disas': ['Disas_34.csv', 'Disas_37.csv', 'Disas_38.csv', 'Disas_36.csv', 'Disas_35.csv'],
    'Re-Assoc': ['(Re)Assoc_24.csv', '(Re)Assoc_29.csv', '(Re)Assoc_25.csv', '(Re)Assoc_28.csv', '(Re)Assoc_34.csv'],
    'Rogue_AP': ['RogueAP_29.csv', 'RogueAP_35.csv', 'RogueAP_38.csv', 'RogueAP_37.csv', 'RogueAP_34.csv'],
    'Krack': ['Krack_26.csv', 'Krack_27.csv', 'Krack_25.csv', 'Krack_28.csv'],
    'Kr00k': ['Kr00k_40.csv', 'Kr00k_39.csv', 'Kr00k_51.csv', 'Kr00k_37.csv', 'Kr00k_54.csv'],
    'SSH': ['SSH_31.csv', 'SSH_45.csv', 'SSH_44.csv', 'SSH_28.csv', 'SSH_30.csv'],
    'Botnet': ['Botnet_25.csv', 'Botnet_24.csv', 'Botnet_56.csv', 'Botnet_63.csv', 'Botnet_53.csv'],
    'Malware': ['Malware_40.csv', 'Malware_45.csv', 'Malware_41.csv', 'Malware_43.csv', 'Malware_36.csv'],
    'SQL_Injection': ['SQL_Injection_36.csv', 'SQL_Injection_46.csv', 'SQL_Injection_37.csv', 'SQL_Injection_48.csv', 'SQL_Injection_42.csv'],
    'SSDP': ['SSDP_161.csv', 'SSDP_158.csv', 'SSDP_159.csv', 'SSDP_157.csv', 'SSDP_160.csv'],
    'Evil_Twin': ['Evil_Twin_42.csv', 'Evil_Twin_35.csv', 'Evil_Twin_53.csv', 'Evil_Twin_43.csv', 'Evil_Twin_50.csv'],
    'Website_spoofing': ['Website_spoofing_52.csv', 'Website_spoofing_32.csv', 'Website_spoofing_34.csv', 'Website_spoofing_44.csv', 'Website_spoofing_48.csv'],
}
mac_cols = ['wlan.sa', 'wlan.da', 'wlan.ta', 'wlan.ra', 'wlan.bssid']
role_cols_map = {'wlan.sa': 0, 'wlan.da': 1, 'wlan.ta': 2, 'wlan.ra': 3, 'wlan.bssid': 4}
reference = {
    'Malware': '00:0c:29:cf:08:aa', 'SSH': '00:0c:29:1d:70:f5',
    'Botnet': '04:ed:33:e0:24:82', 'Krack': '94:e9:79:82:c5:77',
    'Website_spoofing': '04:ed:33:e0:24:82', 'SSDP': '04:ed:33:e0:24:82',
}
reference_sum = {'Evil_Twin': '0c:9d:92:54:fe:35'}

def read_csv_safe(path, usecols=None):
    for attempt in range(3):
        try:
            return pd.read_csv(path, usecols=usecols)
        except OSError:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(3)
    raise RuntimeError(f'Echec de lecture : {path}')

mac_slices = {}
for attack_name, files in known_files_5.items():
    folder_path = os.path.join(base_path, folder_names[attack_name])
    for f in files:
        file_id = f'{attack_name}__{f}'
        if file_id not in fichiers_test_set:
            continue
        df_raw = read_csv_safe(os.path.join(folder_path, f)).reset_index(drop=True)
        attack_idx = df_raw.index[df_raw['Label'] != 'Normal']
        if len(attack_idx) == 0:
            del df_raw; gc.collect(); continue
        start = max(0, attack_idx.min() - 100)
        end = min(len(df_raw), attack_idx.max() + 100)
        cols_present = [c for c in mac_cols if c in df_raw.columns] + ['Label']
        df_mac_slice = df_raw.iloc[start:end][cols_present].reset_index(drop=True)
        df_mac_slice['source_file'] = file_id
        mac_slices[file_id] = df_mac_slice
        del df_raw; gc.collect()

df_mac_combined = pd.concat(mac_slices.values(), ignore_index=True)
del mac_slices
gc.collect()

with open(f'{checkpoint_dir_v4}/le_gnn_v4.pkl', 'rb') as f:
    le_gnn = pickle.load(f)['le']

graph_list, graph_macs, graph_labels, graph_file, graph_pos = [], [], [], [], []
for file_id, group in df_mac_combined.groupby('source_file', sort=False):
    mac_cols_present = [c for c in mac_cols if c in group.columns]
    attack_label = file_id.split('__')[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        window_df = group.iloc[s:s + window_size]
        edges = set()
        role_counts = {}
        for _, row in window_df.iterrows():
            present = [(c, row[c]) for c in mac_cols_present if pd.notna(row[c])]
            addrs = [a for _, a in present]
            for col, mac in present:
                if mac not in role_counts:
                    role_counts[mac] = [0, 0, 0, 0, 0]
                role_counts[mac][role_cols_map[col]] += 1
            if len(addrs) >= 2:
                edges.add(tuple(sorted(addrs[:2])))
        node_list = sorted(role_counts.keys()) if role_counts else ['isole']
        if 'isole' in node_list:
            role_counts['isole'] = [0, 0, 0, 0, 0]
        node_idx = {mac: idx for idx, mac in enumerate(node_list)}
        edge_index_list = []
        for a, b in edges:
            edge_index_list.append([node_idx[a], node_idx[b]])
            edge_index_list.append([node_idx[b], node_idx[a]])
        x = torch.tensor([role_counts[mac] for mac in node_list], dtype=torch.float)
        edge_index = (torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
                      if edge_index_list else torch.zeros((2, 0), dtype=torch.long))
        window_labels = window_df['Label'].values
        label_name = attack_label if (window_labels != 'Normal').any() else 'Normal'
        graph_list.append(Data(x=x, edge_index=edge_index))
        graph_macs.append(node_list)
        graph_labels.append(label_name)
        graph_file.append(file_id)
        graph_pos.append(i)
del df_mac_combined
gc.collect()
print("Graphes des fichiers de test :", len(graph_list))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class GCN_Mean(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim); self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index)); x = F.relu(self.conv2(x, edge_index))
        return self.lin(global_mean_pool(x, batch))
class GCN_Sum(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=32):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim); self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, num_classes)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index)); x = F.relu(self.conv2(x, edge_index))
        return self.lin(global_add_pool(x, batch))

gnn_mean = GCN_Mean(5, len(le_gnn.classes_)).to(device)
gnn_mean.load_state_dict(torch.load(f'{checkpoint_dir_v4}/gnn_v4.pt', map_location=device)); gnn_mean.eval()
gnn_sum = GCN_Sum(5, len(le_gnn.classes_)).to(device)
gnn_sum.load_state_dict(torch.load(f'{checkpoint_dir_v4}/gnn_sum_v4.pt', map_location=device)); gnn_sum.eval()

def proba_model(model, data, cls_idx):
    with torch.no_grad():
        d = data.to(device)
        batch = torch.zeros(d.x.size(0), dtype=torch.long, device=device)
        return F.softmax(model(d.x, d.edge_index, batch), dim=1)[0, cls_idx].item()

explainer = Explainer(
    model=gnn_mean, algorithm=GNNExplainer(epochs=100),
    explanation_type='model', node_mask_type='object', edge_mask_type=None,
    model_config=dict(mode='multiclass_classification', task_level='graph', return_type='raw'),
)

all_labels_rca = list(reference.keys()) + list(reference_sum.keys())
rca_rows = []
for idx in range(len(graph_list)):
    label = graph_labels[idx]
    if label not in all_labels_rca:
        continue
    data = graph_list[idx]; macs = graph_macs[idx]
    n_nodes = data.x.size(0)
    if n_nodes < 2:
        continue
    ref_mac = reference.get(label) or reference_sum.get(label)
    model_use = gnn_sum if label in reference_sum else gnn_mean
    cls_idx = int(le_gnn.transform([label])[0])
    all_proba = [proba_model(model_use, data, c) for c in range(len(le_gnn.classes_))]
    pred0 = le_gnn.inverse_transform([int(np.argmax(all_proba))])[0]
    if pred0 != label:
        continue
    if label in reference_sum:
        p0 = all_proba[cls_idx]
        scores = []
        for node in range(n_nodes):
            keep = [i for i in range(n_nodes) if i != node]
            remap = {old: new for new, old in enumerate(keep)}
            x2 = data.x[keep]
            mask_edge = [(e0.item() in remap and e1.item() in remap)
                         for e0, e1 in zip(data.edge_index[0], data.edge_index[1])]
            ei = (torch.tensor([[remap[e.item()] for e in data.edge_index[:, mask_edge][0]],
                                [remap[e.item()] for e in data.edge_index[:, mask_edge][1]]], dtype=torch.long)
                  if data.edge_index.size(1) > 0 and any(mask_edge) else torch.zeros((2, 0), dtype=torch.long))
            scores.append(p0 - proba_model(gnn_sum, Data(x=x2, edge_index=ei), cls_idx))
        scores = np.array(scores)
    else:
        batch = torch.zeros(n_nodes, dtype=torch.long, device=device)
        explanation = explainer(x=data.x.to(device), edge_index=data.edge_index.to(device),
                                batch=batch, target=torch.tensor([cls_idx], device=device))
        scores = explanation.node_mask.detach().cpu().numpy()
        if scores.ndim > 1:
            scores = scores.sum(axis=1)
    device_designe = macs[int(np.argmax(scores))]
    rca_rows.append({'source_file': graph_file[idx], 'position': graph_pos[idx], 'attaque': label,
                     'appareil_designe': device_designe, 'appareil_correct': device_designe == ref_mac})
    if len(rca_rows) % 200 == 0:
        print(f"  {len(rca_rows)} fenetres RCA traitees...")

rca_df_env = pd.DataFrame(rca_rows)
rca_df_env.to_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')
print(f"\nRCA recalcule sur les fichiers de test de l'environnement : {len(rca_df_env)} fenetres, precision {rca_df_env['appareil_correct'].mean():.4f}")
print("Fichier sauvegarde, remplace la version precedente.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fichiers de test (environnement) : 13
Graphes des fichiers de test : 30080
  200 fenetres RCA traitees...
  400 fenetres RCA traitees...
  600 fenetres RCA traitees...
  800 fenetres RCA traitees...
  1000 fenetres RCA traitees...
  1200 fenetres RCA traitees...
  1400 fenetres RCA traitees...
  1600 fenetres RCA traitees...
  1800 fenetres RCA traitees...
  2000 fenetres RCA traitees...
  2200 fenetres RCA traitees...
  2400 fenetres RCA traitees...
  2600 fenetres RCA traitees...
  2800 fenetres RCA traitees...
  3000 fenetres RCA traitees...
  3200 fenetres RCA traitees...
  3400 fenetres RCA traitees...
  3600 fenetres RCA traitees...
  3800 fenetres RCA traitees...
  4000 fenetres RCA traitees...
  4200 fenetres RCA traitees...
  4400 fenetres RCA traitees...
  4600 fenetres RCA traitees...
  4800 fenetres RCA traitees...
  5000 fenetres RCA traitees...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
fenetre_memoire = 200

env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_v4.parquet')
rca_df = pd.read_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')

rca_lookup = rca_df.set_index(['source_file', 'position'])['appareil_designe'].to_dict()

env_df = env_df.sort_values(['source_file', 'position']).reset_index(drop=True)

incidents_recents = []
repetition_appareil = []
appareil_present_calcule = []

for file_id, group in env_df.groupby('source_file', sort=False):
    positions = group['position'].values
    historique_incidents = []
    historique_appareils = []
    for pos in positions:
        fenetre_debut = pos - fenetre_memoire
        while historique_incidents and historique_incidents[0] < fenetre_debut:
            historique_incidents.pop(0)
            historique_appareils.pop(0)

        incidents_recents.append(len(historique_incidents))
        appareil_ici = rca_lookup.get((file_id, pos), None)
        appareil_present_calcule.append(appareil_ici)
        if appareil_ici is not None and len(historique_appareils) > 0:
            repetition_appareil.append(historique_appareils.count(appareil_ici))
        else:
            repetition_appareil.append(0)

        row_mask = (group['position'] == pos)
        label_ici = group.loc[row_mask, 'vraie_etiquette'].iloc[0]
        if label_ici != 'Normal':
            historique_incidents.append(pos)
            historique_appareils.append(appareil_ici)

env_df['incidents_recents_200'] = incidents_recents
env_df['repetition_appareil'] = repetition_appareil
env_df['appareil_designe'] = appareil_present_calcule

print("Fenetres avec RCA calcule :", env_df['appareil_designe'].notna().sum())
print("\nDistribution des incidents recents :")
print(env_df['incidents_recents_200'].describe())
print("\nDistribution de la repetition d'appareil :")
print(env_df['repetition_appareil'].describe())

env_df.to_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')
print("\nEtat MAPE-K sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fenetres avec RCA calcule : 6924

Distribution des incidents recents :
count    30080.000000
mean        96.246775
std         74.405224
min          0.000000
25%         23.000000
50%         80.000000
75%        177.000000
max        200.000000
Name: incidents_recents_200, dtype: float64

Distribution de la repetition d'appareil :
count    30080.000000
mean        13.599967
std         37.400817
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        171.000000
Name: repetition_appareil, dtype: float64

Etat MAPE-K sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')

def construire_etat(row, config):
    etat = []
    if 'confiance' in config:
        etat.append(row['confiance_hybride'])
    if 'indicateur' in config:
        etat.append(1.0 if row['prediction_hybride'] != 'Normal' else 0.0)
    if 'memoire' in config:
        etat.append(min(row['incidents_recents_200'] / 200.0, 1.0))
        etat.append(min(row['repetition_appareil'] / 20.0, 1.0))
    return np.array(etat, dtype=np.float32)

configs_a_tester = {
    'confiance_seule': ['confiance'],
    'confiance_memoire': ['confiance', 'memoire'],
    'indicateur_seul': ['indicateur'],
    'indicateur_memoire': ['indicateur', 'memoire'],
}

def entrainer_agent(domaine_df, config_cles, n_epochs=30, lr=0.05, seed=42):
    rng = np.random.RandomState(seed)
    dim = len(construire_etat(domaine_df.iloc[0], config_cles))
    poids = rng.normal(0, 0.1, dim).astype(np.float32)
    biais = 0.0
    historique_taux_intervention = []

    indices = np.arange(len(domaine_df))
    for epoch in range(n_epochs):
        rng.shuffle(indices)
        interventions_epoch = 0
        for i in indices:
            row = domaine_df.iloc[i]
            etat = construire_etat(row, config_cles)
            score = etat @ poids + biais
            proba_action = 1 / (1 + np.exp(-score))
            action = 1 if rng.random() < proba_action else 0
            interventions_epoch += action

            vrai_label = row['vraie_etiquette']
            recompense = 1.0 if (action == 1) == (vrai_label != 'Normal') else -1.0
            if action == 1 and vrai_label == 'Normal':
                recompense = -2.0

            gradient = (action - proba_action) * recompense
            poids += lr * gradient * etat
            biais += lr * gradient

        historique_taux_intervention.append(interventions_epoch / len(domaine_df))

    return poids, biais, historique_taux_intervention

resultats_configs = {}
stabilite_configs = {}
for nom_config, cles in configs_a_tester.items():
    print(f"\n=== Configuration : {nom_config} ===")
    decisions_par_domaine = {}
    stabilite_par_domaine = {}

    for domaine in ['Wi-Fi', 'Reseau']:
        sous_df = env_df[env_df['domaine'] == domaine].reset_index(drop=True)
        poids, biais, hist_taux = entrainer_agent(sous_df, cles)
        stabilite_par_domaine[domaine] = hist_taux

        actions = []
        for i in range(len(sous_df)):
            etat = construire_etat(sous_df.iloc[i], cles)
            score = etat @ poids + biais
            proba = 1 / (1 + np.exp(-score))
            actions.append(1 if proba >= 0.5 else 0)
        sous_df['action'] = actions
        decisions_par_domaine[domaine] = sous_df

    toutes_decisions = pd.concat(decisions_par_domaine.values(), ignore_index=True)
    vrais_positifs = ((toutes_decisions['action'] == 1) & (toutes_decisions['vraie_etiquette'] != 'Normal')).sum()
    total_attaques = (toutes_decisions['vraie_etiquette'] != 'Normal').sum()
    fausses_alertes = ((toutes_decisions['action'] == 1) & (toutes_decisions['vraie_etiquette'] == 'Normal')).sum()
    total_normal = (toutes_decisions['vraie_etiquette'] == 'Normal').sum()

    taux_detection = vrais_positifs / total_attaques
    taux_fa = fausses_alertes / total_normal

    resultats_configs[nom_config] = {
        'taux_detection': taux_detection, 'taux_fausses_alertes': taux_fa,
    }
    stabilite_configs[nom_config] = stabilite_par_domaine
    print(f"  Taux de detection : {taux_detection:.4f} | Taux de fausses alertes : {taux_fa:.4f}")

    if nom_config == 'indicateur_memoire':
        decisions_wifi = decisions_par_domaine['Wi-Fi'].set_index(['source_file', 'position'])['action']
        decisions_reseau = decisions_par_domaine['Reseau'].set_index(['source_file', 'position'])['action']
        communes = decisions_wifi.index.intersection(decisions_reseau.index)
        double_intervention = ((decisions_wifi.loc[communes] == 1) & (decisions_reseau.loc[communes] == 1)).sum()
        print(f"  Double intervention : {double_intervention} / {len(communes)}")

print("\n=== Tableau recapitulatif des 4 configurations ===")
print(pd.DataFrame(resultats_configs).T.round(4))

with open(f'{checkpoint_dir_v4}/resultats_auto_guerison_v4.pkl', 'wb') as f:
    pickle.dump({'resultats': resultats_configs, 'stabilite': stabilite_configs}, f)
print("\nResultats et courbes de stabilite sauvegardes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Configuration : confiance_seule ===
  Taux de detection : 0.0000 | Taux de fausses alertes : 0.0000

=== Configuration : confiance_memoire ===
  Taux de detection : 0.7414 | Taux de fausses alertes : 0.0534

=== Configuration : indicateur_seul ===
  Taux de detection : 0.9794 | Taux de fausses alertes : 0.0315

=== Configuration : indicateur_memoire ===
  Taux de detection : 0.9794 | Taux de fausses alertes : 0.0315
  Double intervention : 0 / 0

=== Tableau recapitulatif des 4 configurations ===
                    taux_detection  taux_fausses_alertes
confiance_seule             0.0000                0.0000
confiance_memoire           0.7414                0.0534
indicateur_seul             0.9794                0.0315
indicateur_memoire          0.9794                0.0315

Resultats et courbes de stabilite sauvegardes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')

def construire_etat(row, config):
    etat = []
    if 'confiance' in config:
        etat.append(row['confiance_hybride'])
    if 'indicateur' in config:
        etat.append(1.0 if row['prediction_hybride'] != 'Normal' else 0.0)
    if 'memoire' in config:
        etat.append(min(row['incidents_recents_200'] / 200.0, 1.0))
        etat.append(min(row['repetition_appareil'] / 20.0, 1.0))
    return np.array(etat, dtype=np.float32)

def entrainer_agent(domaine_df, config_cles, n_epochs=30, lr=0.05, seed=42):
    rng = np.random.RandomState(seed)
    dim = len(construire_etat(domaine_df.iloc[0], config_cles))
    poids = rng.normal(0, 0.1, dim).astype(np.float32)
    biais = 0.0
    indices = np.arange(len(domaine_df))
    for epoch in range(n_epochs):
        rng.shuffle(indices)
        for i in indices:
            row = domaine_df.iloc[i]
            etat = construire_etat(row, config_cles)
            score = etat @ poids + biais
            proba_action = 1 / (1 + np.exp(-score))
            action = 1 if rng.random() < proba_action else 0
            vrai_label = row['vraie_etiquette']
            recompense = 1.0 if (action == 1) == (vrai_label != 'Normal') else -1.0
            if action == 1 and vrai_label == 'Normal':
                recompense = -2.0
            gradient = (action - proba_action) * recompense
            poids += lr * gradient * etat
            biais += lr * gradient
    return poids, biais

def predire(df, poids, biais, config_cles):
    actions = []
    for i in range(len(df)):
        etat = construire_etat(df.iloc[i], config_cles)
        score = etat @ poids + biais
        proba = 1 / (1 + np.exp(-score))
        actions.append(1 if proba >= 0.5 else 0)
    return actions

config_cles = ['indicateur', 'memoire']  # la configuration retenue : indicateur + memoire

poids_wifi, biais_wifi = entrainer_agent(env_df[env_df['domaine'] == 'Wi-Fi'].reset_index(drop=True), config_cles)
poids_reseau, biais_reseau = entrainer_agent(env_df[env_df['domaine'] == 'Reseau'].reset_index(drop=True), config_cles)

env_eval = env_df.copy().reset_index(drop=True)
env_eval['action_wifi'] = predire(env_eval, poids_wifi, biais_wifi, config_cles)
env_eval['action_reseau'] = predire(env_eval, poids_reseau, biais_reseau, config_cles)

double_intervention = ((env_eval['action_wifi'] == 1) & (env_eval['action_reseau'] == 1)).sum()
print(f"Double intervention (les deux agents interviennent sur la meme fenetre) : {double_intervention} / {len(env_eval)}")

print("\nRepartition :")
print(f"  Seul Wi-Fi intervient     : {((env_eval['action_wifi']==1)&(env_eval['action_reseau']==0)).sum()}")
print(f"  Seul Reseau intervient    : {((env_eval['action_wifi']==0)&(env_eval['action_reseau']==1)).sum()}")
print(f"  Aucun des deux n'intervient : {((env_eval['action_wifi']==0)&(env_eval['action_reseau']==0)).sum()}")

env_eval.to_parquet(f'{checkpoint_dir_v4}/resultats_auto_guerison_double_intervention_v4.parquet')
print("\nResultat sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Double intervention (les deux agents interviennent sur la meme fenetre) : 15322 / 30080

Repartition :
  Seul Wi-Fi intervient     : 0
  Seul Reseau intervient    : 0
  Aucun des deux n'intervient : 14758

Resultat sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
fenetre_memoire = 200

env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_v4.parquet')
rca_df = pd.read_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')
rca_lookup = rca_df.set_index(['source_file', 'position'])['appareil_designe'].to_dict()

env_df = env_df.sort_values(['source_file', 'position']).reset_index(drop=True)

incidents_recents = []
repetition_appareil = []
appareil_present_calcule = []

for file_id, group in env_df.groupby('source_file', sort=False):
    positions = group['position'].values
    historique_incidents = []
    historique_appareils = []  # ne contient desormais QUE de vraies adresses, jamais None
    for pos in positions:
        fenetre_debut = pos - fenetre_memoire
        while historique_incidents and historique_incidents[0] < fenetre_debut:
            historique_incidents.pop(0)
            historique_appareils.pop(0)

        incidents_recents.append(len(historique_incidents))
        appareil_ici = rca_lookup.get((file_id, pos), None)
        appareil_present_calcule.append(appareil_ici)
        # CORRECTION : on ne compte une repetition que si l'appareil est reellement connu
        if appareil_ici is not None:
            repetition_appareil.append(historique_appareils.count(appareil_ici))
        else:
            repetition_appareil.append(0)

        row_mask = (group['position'] == pos)
        label_ici = group.loc[row_mask, 'vraie_etiquette'].iloc[0]
        if label_ici != 'Normal':
            historique_incidents.append(pos)
            if appareil_ici is not None:          # CORRECTION : on n'ajoute plus jamais None
                historique_appareils.append(appareil_ici)
            else:
                historique_incidents_len_fix = None  # rien a ajouter cote appareils, listes restent desynchronisees volontairement

# Reecriture propre : on synchronise differemment pour eviter tout desalignement entre les deux listes
incidents_recents, repetition_appareil, appareil_present_calcule = [], [], []
for file_id, group in env_df.groupby('source_file', sort=False):
    positions = group['position'].values
    historique_incidents_pos = []     # positions des incidents recents
    historique_incidents_app = []     # appareil correspondant, ou None
    for pos in positions:
        fenetre_debut = pos - fenetre_memoire
        while historique_incidents_pos and historique_incidents_pos[0] < fenetre_debut:
            historique_incidents_pos.pop(0)
            historique_incidents_app.pop(0)

        incidents_recents.append(len(historique_incidents_pos))
        appareil_ici = rca_lookup.get((file_id, pos), None)
        appareil_present_calcule.append(appareil_ici)
        if appareil_ici is not None:
            # on ne compte que les occurrences passees ou l'appareil etait reellement connu ET identique
            repetition_appareil.append(sum(1 for a in historique_incidents_app if a == appareil_ici))
        else:
            repetition_appareil.append(0)

        row_mask = (group['position'] == pos)
        label_ici = group.loc[row_mask, 'vraie_etiquette'].iloc[0]
        if label_ici != 'Normal':
            historique_incidents_pos.append(pos)
            historique_incidents_app.append(appareil_ici)  # garde None explicitement, mais ne sera jamais compte comme un match

env_df['incidents_recents_200'] = incidents_recents
env_df['repetition_appareil'] = repetition_appareil
env_df['appareil_designe'] = appareil_present_calcule

print("Fenetres avec RCA calcule :", env_df['appareil_designe'].notna().sum())
print("\nDistribution de la repetition d'appareil (corrigee, sans les faux-matchs sur None) :")
print(env_df['repetition_appareil'].describe())

env_df.to_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')
print("\nEtat MAPE-K corrige sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


IndexError: pop from empty list

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
fenetre_memoire = 200

env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_v4.parquet')
rca_df = pd.read_parquet(f'{checkpoint_dir_v4}/rca_pour_auto_guerison_v4.parquet')
rca_lookup = rca_df.set_index(['source_file', 'position'])['appareil_designe'].to_dict()

env_df = env_df.sort_values(['source_file', 'position']).reset_index(drop=True)

incidents_recents, repetition_appareil, appareil_present_calcule = [], [], []

for file_id, group in env_df.groupby('source_file', sort=False):
    positions = group['position'].values
    historique_incidents_pos = []   # positions des incidents recents, toujours alimentee
    historique_incidents_app = []   # appareil correspondant (peut etre None), toujours de meme longueur que ci-dessus

    for pos in positions:
        fenetre_debut = pos - fenetre_memoire
        while historique_incidents_pos and historique_incidents_pos[0] < fenetre_debut:
            historique_incidents_pos.pop(0)
            historique_incidents_app.pop(0)

        incidents_recents.append(len(historique_incidents_pos))

        appareil_ici = rca_lookup.get((file_id, pos), None)
        appareil_present_calcule.append(appareil_ici)

        if appareil_ici is not None:
            repetition_appareil.append(sum(1 for a in historique_incidents_app if a == appareil_ici))
        else:
            repetition_appareil.append(0)

        row_mask = (group['position'] == pos)
        label_ici = group.loc[row_mask, 'vraie_etiquette'].iloc[0]
        if label_ici != 'Normal':
            historique_incidents_pos.append(pos)
            historique_incidents_app.append(appareil_ici)  # les deux listes restent toujours synchronisees, meme si appareil_ici est None

env_df['incidents_recents_200'] = incidents_recents
env_df['repetition_appareil'] = repetition_appareil
env_df['appareil_designe'] = appareil_present_calcule

print("Fenetres avec RCA calcule :", env_df['appareil_designe'].notna().sum())
print("\nDistribution des incidents recents :")
print(env_df['incidents_recents_200'].describe())
print("\nDistribution de la repetition d'appareil (corrigee) :")
print(env_df['repetition_appareil'].describe())

env_df.to_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')
print("\nEtat MAPE-K corrige sauvegarde.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fenetres avec RCA calcule : 6924

Distribution des incidents recents :
count    30080.000000
mean        96.246775
std         74.405224
min          0.000000
25%         23.000000
50%         80.000000
75%        177.000000
max        200.000000
Name: incidents_recents_200, dtype: float64

Distribution de la repetition d'appareil (corrigee) :
count    30080.000000
mean        13.599967
std         37.400817
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        171.000000
Name: repetition_appareil, dtype: float64

Etat MAPE-K corrige sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')

def construire_etat(row, config):
    etat = []
    if 'confiance' in config:
        etat.append(row['confiance_hybride'])
    if 'indicateur' in config:
        etat.append(1.0 if row['prediction_hybride'] != 'Normal' else 0.0)
    if 'memoire' in config:
        etat.append(min(row['incidents_recents_200'] / 200.0, 1.0))
        etat.append(min(row['repetition_appareil'] / 20.0, 1.0))
    return np.array(etat, dtype=np.float32)

def entrainer_agent(domaine_df, config_cles, n_epochs=30, lr=0.05, seed=42):
    rng = np.random.RandomState(seed)
    dim = len(construire_etat(domaine_df.iloc[0], config_cles))
    poids = rng.normal(0, 0.1, dim).astype(np.float32)
    biais = 0.0
    indices = np.arange(len(domaine_df))
    for epoch in range(n_epochs):
        rng.shuffle(indices)
        for i in indices:
            row = domaine_df.iloc[i]
            etat = construire_etat(row, config_cles)
            score = etat @ poids + biais
            proba_action = 1 / (1 + np.exp(-score))
            action = 1 if rng.random() < proba_action else 0
            vrai_label = row['vraie_etiquette']
            recompense = 1.0 if (action == 1) == (vrai_label != 'Normal') else -1.0
            if action == 1 and vrai_label == 'Normal':
                recompense = -2.0
            gradient = (action - proba_action) * recompense
            poids += lr * gradient * etat
            biais += lr * gradient
    return poids, biais

configs_a_tester = {
    'confiance_seule': ['confiance'],
    'confiance_memoire': ['confiance', 'memoire'],
    'indicateur_seul': ['indicateur'],
    'indicateur_memoire': ['indicateur', 'memoire'],
}

agents_entraines = {}
resultats_configs = {}

for nom_config, cles in configs_a_tester.items():
    poids_wifi, biais_wifi = entrainer_agent(env_df[env_df['domaine'] == 'Wi-Fi'].reset_index(drop=True), cles)
    poids_reseau, biais_reseau = entrainer_agent(env_df[env_df['domaine'] == 'Reseau'].reset_index(drop=True), cles)
    agents_entraines[nom_config] = {'wifi': (poids_wifi, biais_wifi), 'reseau': (poids_reseau, biais_reseau)}

    env_eval = env_df.copy()
    def decider(row):
        poids, biais = (poids_wifi, biais_wifi) if row['domaine'] == 'Wi-Fi' else (poids_reseau, biais_reseau)
        etat = construire_etat(row, cles)
        proba = 1 / (1 + np.exp(-(etat @ poids + biais)))
        return 1 if proba >= 0.5 else 0
    env_eval['action'] = env_eval.apply(decider, axis=1)

    vrais_positifs = ((env_eval['action'] == 1) & (env_eval['vraie_etiquette'] != 'Normal')).sum()
    total_attaques = (env_eval['vraie_etiquette'] != 'Normal').sum()
    fausses_alertes = ((env_eval['action'] == 1) & (env_eval['vraie_etiquette'] == 'Normal')).sum()
    total_normal = (env_eval['vraie_etiquette'] == 'Normal').sum()

    resultats_configs[nom_config] = {
        'taux_detection': vrais_positifs / total_attaques,
        'taux_fausses_alertes': fausses_alertes / total_normal,
    }
    print(f"{nom_config:20s} : detection={resultats_configs[nom_config]['taux_detection']:.4f} | "
          f"fausses_alertes={resultats_configs[nom_config]['taux_fausses_alertes']:.4f}")

    if nom_config == 'indicateur_memoire':
        env_eval.to_parquet(f'{checkpoint_dir_v4}/decisions_correctives_v4.parquet')

print("\n=== Tableau recapitulatif ===")
print(pd.DataFrame(resultats_configs).T.round(4))

with open(f'{checkpoint_dir_v4}/agents_decisions_correctives_v4.pkl', 'wb') as f:
    pickle.dump(agents_entraines, f)
print("\nAgents et decisions sauvegardes (decisions_correctives_v4.parquet contient la config retenue, indicateur_memoire).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
confiance_seule      : detection=0.0000 | fausses_alertes=0.0000
confiance_memoire    : detection=0.7414 | fausses_alertes=0.0534
indicateur_seul      : detection=0.9794 | fausses_alertes=0.0315
indicateur_memoire   : detection=0.9794 | fausses_alertes=0.0315

=== Tableau recapitulatif ===
                    taux_detection  taux_fausses_alertes
confiance_seule             0.0000                0.0000
confiance_memoire           0.7414                0.0534
indicateur_seul             0.9794                0.0315
indicateur_memoire          0.9794                0.0315

Agents et decisions sauvegardes (decisions_correctives_v4.parquet contient la config retenue, indicateur_memoire).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

env_etat = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')

base_connaissances = env_etat[[
    'source_file', 'position', 'domaine', 'vraie_etiquette',
    'prediction_hybride', 'confiance_hybride',
    'appareil_designe',
    'incidents_recents_200', 'repetition_appareil',
]].copy()

# === ENTREE 1 : Detection (predictions recentes) ===
base_connaissances = base_connaissances.rename(columns={
    'incidents_recents_200': 'memoire_court_terme_incidents',
    'repetition_appareil': 'memoire_court_terme_repetition_appareil',
})

# === ENTREE 2 : Diagnostic (appareils designes par le RCA) ===
# deja presente via 'appareil_designe', reprise telle quelle

# === ENTREE 3 : Execution (resultat de l'action) — CONCEPTUELLE, aucune donnee reelle ===
base_connaissances['resultat_action_execution'] = np.nan  # colonne presente, jamais alimentee : module non implemente

print("=== Base de connaissances - entrees ===")
print("1. Detection       : predictions recentes (colonne prediction_hybride)        -> reelle")
print("2. Diagnostic       : appareils designes par le RCA (colonne appareil_designe) -> reelle")
print("3. Execution        : resultat de l'action (colonne resultat_action_execution) -> CONCEPTUELLE, toujours vide")
print(f"   Verification : {base_connaissances['resultat_action_execution'].notna().sum()} valeurs non vides (attendu : 0)")

# === SORTIE 1 : vers Decisions correctives (contexte memoire) — REELLE ===
sortie_decisions = base_connaissances[[
    'source_file', 'position', 'domaine',
    'memoire_court_terme_incidents', 'memoire_court_terme_repetition_appareil',
]].copy()

# === SORTIE 2 : vers Detection/Diagnostic (reentrainement) — CONCEPTUELLE ===
sortie_reentrainement = pd.DataFrame({
    'description': ['Reentrainement periodique des modeles de Detection et de Diagnostic a partir de '
                     'l historique persistant de la Base de connaissances'],
    'statut': ['conceptuel, non implemente dans ce travail'],
    'mecanisme_construit': [False],
})

print("\n=== Base de connaissances - sorties ===")
print("1. Vers Decisions correctives : contexte memoire par fenetre -> reelle,", len(sortie_decisions), "lignes")
print("2. Vers Detection/Diagnostic   : reentrainement               -> CONCEPTUELLE, aucun mecanisme construit")

base_connaissances.to_parquet(f'{checkpoint_dir_v4}/base_connaissances_v4.parquet')
sortie_decisions.to_parquet(f'{checkpoint_dir_v4}/base_connaissances_sortie_decisions_v4.parquet')
sortie_reentrainement.to_csv(f'{checkpoint_dir_v4}/base_connaissances_sortie_reentrainement_v4.csv', index=False)

print("\nTrois fichiers sauvegardes : le bloc complet, la sortie reelle vers Decisions correctives,")
print("et la sortie conceptuelle vers reentrainement (documentee, non calculee).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Base de connaissances - entrees ===
1. Detection       : predictions recentes (colonne prediction_hybride)        -> reelle
2. Diagnostic       : appareils designes par le RCA (colonne appareil_designe) -> reelle
3. Execution        : resultat de l'action (colonne resultat_action_execution) -> CONCEPTUELLE, toujours vide
   Verification : 0 valeurs non vides (attendu : 0)

=== Base de connaissances - sorties ===
1. Vers Decisions correctives : contexte memoire par fenetre -> reelle, 30080 lignes
2. Vers Detection/Diagnostic   : reentrainement               -> CONCEPTUELLE, aucun mecanisme construit

Trois fichiers sauvegardes : le bloc complet, la sortie reelle vers Decisions correctives,
et la sortie conceptuelle vers reentrainement (documentee, non calculee).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle
import numpy as np
import pandas as pd

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

env_df = pd.read_parquet(f'{checkpoint_dir_v4}/env_auto_guerison_etat_v4.parquet')
with open(f'{checkpoint_dir_v4}/agents_decisions_correctives_v4.pkl', 'rb') as f:
    agents = pickle.load(f)['indicateur_memoire']

poids_wifi, biais_wifi = agents['wifi']
poids_reseau, biais_reseau = agents['reseau']

def construire_etat(row):
    return np.array([
        1.0 if row['prediction_hybride'] != 'Normal' else 0.0,
        min(row['incidents_recents_200'] / 200.0, 1.0),
        min(row['repetition_appareil'] / 20.0, 1.0),
    ], dtype=np.float32)

def decider(etat, poids, biais):
    proba = 1 / (1 + np.exp(-(etat @ poids + biais)))
    return 1 if proba >= 0.5 else 0

actions_wifi = []
actions_reseau = []
for _, row in env_df.iterrows():
    etat = construire_etat(row)
    actions_wifi.append(decider(etat, poids_wifi, biais_wifi))
    actions_reseau.append(decider(etat, poids_reseau, biais_reseau))

env_df['action_agent_wifi'] = actions_wifi
env_df['action_agent_reseau'] = actions_reseau

# Verification du mecanisme de routage reel : seul l'agent du domaine concerne
# recoit effectivement la fenetre dans le systeme deploye
env_df['agent_reellement_sollicite'] = env_df.apply(
    lambda r: r['action_agent_wifi'] if r['domaine'] == 'Wi-Fi' else r['action_agent_reseau'], axis=1
)

print("=== Verification : routage par domaine ===")
print("Fenetres Wi-Fi traitees par l'agent Wi-Fi uniquement :",
      (env_df['domaine'] == 'Wi-Fi').sum())
print("Fenetres Reseau traitees par l'agent Reseau uniquement :",
      (env_df['domaine'] == 'Reseau').sum())

double_intervention_reelle = 0  # par construction du routage, calcule explicitement ci-dessous pour verification
conflits_potentiels = ((env_df['action_agent_wifi'] == 1) & (env_df['action_agent_reseau'] == 1)).sum()

print(f"\n=== Resultat ===")
print(f"Double intervention dans le systeme reellement deploye (un seul agent par fenetre, selon le domaine) : "
      f"{double_intervention_reelle} / {len(env_df)}")
print(f"\nPour information seulement, si les deux agents evaluaient systematiquement TOUTES les fenetres "
      f"sans routage (scenario hors architecture reelle) : {conflits_potentiels} / {len(env_df)} auraient "
      f"declenche les deux en meme temps")

env_df.to_parquet(f'{checkpoint_dir_v4}/verification_double_intervention_v4.parquet')
print("\nResultat sauvegarde.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Verification : routage par domaine ===
Fenetres Wi-Fi traitees par l'agent Wi-Fi uniquement : 16249
Fenetres Reseau traitees par l'agent Reseau uniquement : 13831

=== Resultat ===
Double intervention dans le systeme reellement deploye (un seul agent par fenetre, selon le domaine) : 0 / 30080

Pour information seulement, si les deux agents evaluaient systematiquement TOUTES les fenetres sans routage (scenario hors architecture reelle) : 15322 / 30080 auraient declenche les deux en meme temps

Resultat sauvegarde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.simplefilter('ignore')

import pickle, joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tensorflow.keras import models as keras_models
import shap

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

# On reprend exactement le meme decoupage par fichiers deja valide pour le module de decision,
# pour rester coherent avec le reste du memoire plutot que d'en inventer un nouveau
fichiers_train = pd.read_csv(f'{checkpoint_dir_v4}/fichiers_train_auto_guerison_v4.csv')['0'].tolist()
fichiers_test = pd.read_csv(f'{checkpoint_dir_v4}/fichiers_test_auto_guerison_v4.csv')['0'].tolist()

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']; feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')

def build_windows(df, file_list):
    df_sub = df[df['source_file'].isin(file_list)]
    X_max, X_seq, y = [], [], []
    for file_id, group in df_sub.groupby('source_file', sort=False):
        feats = group[feature_cols].values.astype(np.float32)
        labels = group['Label'].values
        attack = group['source_attack'].iloc[0]
        n_win = len(group) // window_size
        for i in range(n_win):
            s = i * window_size
            window = feats[s:s+window_size]
            wl = labels[s:s+window_size]
            X_max.append(window.max(axis=0))
            X_seq.append(window)
            y.append(attack if (wl != 'Normal').any() else 'Normal')
    return np.array(X_max), np.array(X_seq), le_max.transform(y)

print("Construction des fenetres d'entrainement (fichiers jamais vus au test)...")
X_max_train, X_seq_train, y_train = build_windows(df_encoded, fichiers_train)
print("Construction des fenetres de test (fichiers entiers, jamais vus a l'entrainement)...")
X_max_test, X_seq_test, y_test = build_windows(df_encoded, fichiers_test)
print("Train:", X_max_train.shape, "Test:", X_max_test.shape)

# --- XGBoost reentraine sur les fichiers d'entrainement, SMOTE applique uniquement ici ---
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix

sm = SMOTE(random_state=42)
X_max_train_sm, y_train_sm = sm.fit_resample(X_max_train, y_train)
xgb_file = XGBClassifier(n_estimators=300, max_depth=6, random_state=42, eval_metric='mlogloss')
xgb_file.fit(X_max_train_sm, y_train_sm)

proba_xgb = xgb_file.predict_proba(X_max_test)
pred_xgb = proba_xgb.argmax(axis=1)
acc = accuracy_score(y_test, pred_xgb)
rec = recall_score(y_test, pred_xgb, average='macro')
f1 = f1_score(y_test, pred_xgb, average='macro')
auc = roc_auc_score(y_test, proba_xgb, multi_class='ovr', average='macro')
cm = confusion_matrix(y_test, pred_xgb)

print(f"\n=== XGBoost, reentraine, evalue sur fichiers entierement inedits ===")
print(f"Accuracy={acc:.4f} Rappel_macro={rec:.4f} F1_macro={f1:.4f} ROC_AUC_macro={auc:.4f}")

# --- Fidelite SHAP (k=10), calculee sur ce meme decoupage par fichiers ---
explainer = shap.TreeExplainer(xgb_file)
sample_idx = np.random.RandomState(42).choice(len(X_max_test), size=min(500, len(X_max_test)), replace=False)
shap_values = explainer.shap_values(X_max_test[sample_idx])

def fidelite_top_k(k):
    chutes_reelles, chutes_hasard = [], []
    rng = np.random.RandomState(42)
    for i, idx in enumerate(sample_idx):
        pred_classe = pred_xgb[idx]
        sv = shap_values[pred_classe][i] if isinstance(shap_values, list) else shap_values[i, :, pred_classe]
        top_k_idx = np.argsort(-np.abs(sv))[:k]
        x_modif = X_max_test[idx].copy()
        x_modif[top_k_idx] = 0
        p_avant = proba_xgb[idx, pred_classe]
        p_apres = xgb_file.predict_proba(x_modif.reshape(1,-1))[0, pred_classe]
        chutes_reelles.append(p_avant - p_apres)
        hasard_idx = rng.choice(len(x_modif), size=k, replace=False)
        x_hasard = X_max_test[idx].copy(); x_hasard[hasard_idx] = 0
        p_hasard = xgb_file.predict_proba(x_hasard.reshape(1,-1))[0, pred_classe]
        chutes_hasard.append(p_avant - p_hasard)
    return np.mean(chutes_reelles), np.mean(chutes_hasard)

fid10, hasard10 = fidelite_top_k(10)
print(f"\n=== Fidelite SHAP (k=10), sous decoupage par fichiers ===")
print(f"Chute reelle={fid10:.4f} | Chute au hasard={hasard10:.4f}")

print("\nMatrice de confusion (lignes=vrai, colonnes=predit) :")
print(cm)

pd.DataFrame(cm, index=le_max.classes_, columns=le_max.classes_).to_csv(f'{checkpoint_dir_v4}/matrice_confusion_fichiers_v4.csv')
print("\nMatrice sauvegardee.")

Mounted at /content/drive
Construction des fenetres d'entrainement (fichiers jamais vus au test)...
Construction des fenetres de test (fichiers entiers, jamais vus a l'entrainement)...
Train: (124526, 31) Test: (30080, 31)

=== XGBoost, reentraine, evalue sur fichiers entierement inedits ===
Accuracy=0.8206 Rappel_macro=0.6489 F1_macro=0.6771 ROC_AUC_macro=0.9732

=== Fidelite SHAP (k=10), sous decoupage par fichiers ===
Chute reelle=0.3851 | Chute au hasard=0.2535

Matrice de confusion (lignes=vrai, colonnes=predit) :
[[  554     0     1     5     0     0    62   413     1     0     0     0
      0     0]
 [    1   620    15     0    35     0     0   184    28     0     0     0
      1     1]
 [    2   377  1059     0   266     1     0   185    64     0     9     0
      1     2]
 [    2     0     0   703     0     0     0   347     0     0     1     0
      1     0]
 [    0    33  1018     0  1275     1     0   117     5     2     2     0
      1     0]
 [    0     0     0     3     

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pickle, joblib
from tensorflow.keras import models as keras_models
import torch

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

print("=== Taille de fenetre ===")
print("window_size =", 20, "(fixe dans tout le pipeline)")

print("\n=== XGBoost ===")
xgb_model = joblib.load(f'{checkpoint_dir_v4}/xgb_combo_v4.joblib')
print(xgb_model.get_params())

print("\n=== LSTM ===")
lstm_model = keras_models.load_model(f'{checkpoint_dir_v4}/lstm_v4.keras')
lstm_model.summary()

print("\n=== GNN ===")
gnn_state = torch.load(f'{checkpoint_dir_v4}/gnn_v4.pt', map_location='cpu')
for k, v in gnn_state.items():
    print(k, tuple(v.shape))

print("\n=== Autoencodeur ===")
ae_model = keras_models.load_model(f'{checkpoint_dir_v4}/autoencoder_v4.keras')
ae_model.summary()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Taille de fenetre ===
window_size = 20 (fixe dans tout le pipeline)

=== XGBoost ===
{'objective': 'multi:softprob', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'mlogloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None,

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        24,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 14)             │           462 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,356 (317.80 KB)

 Trainable params: 27,118 (105.93 KB)

 Non-trainable params: 0 (0.00 B)

... (4 outputs left out)

 Total params: 52,607 (205.50 KB)

 Trainable params: 17,535 (68.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 35,072 (137.00 KB)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.simplefilter('ignore')
import pickle, joblib
import numpy as np
import pandas as pd
from tensorflow.keras import models as keras_models
from sklearn.metrics import precision_score, recall_score

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

data_hybrid = np.load(f'{checkpoint_dir_v4}/hybrid_proba_v4.npz')
hybrid_proba = 0.4*data_hybrid['xgb_proba'] + 0.4*data_hybrid['lstm_proba'] + 0.2*data_hybrid['gnn_proba']
S_sup = hybrid_proba.max(axis=1)
y_test = data_hybrid['y_test']

ae_err = np.load(f'{checkpoint_dir_v4}/scaler_autoencoder_v4.pkl', allow_pickle=True)
# on recupere l'erreur de reconstruction deja calculee a l'entrainement du module 3
with open(f'{checkpoint_dir_v4}/scaler_autoencoder_v4.pkl', 'rb') as f:
    ae_saved = pickle.load(f)
err_test = ae_saved.get('err_test', ae_saved.get('err_normal_test'))

vraie_attaque = (y_test != ae_saved.get('label_normal_idx', 0))

print("=== Balayage reel de plusieurs couples de seuils (confiance, erreur) ===")
resultats = []
for tau_conf in [0.3, 0.4, 0.5, 0.6]:
    for tau_err_pct in [75, 85, 90, 95]:
        tau_err = np.percentile(err_test, tau_err_pct)
        signal = (S_sup < tau_conf) & (err_test > tau_err)
        n_signalees = signal.sum()
        if n_signalees == 0:
            continue
        precision = (signal & vraie_attaque).sum() / n_signalees
        rappel = (signal & vraie_attaque).sum() / vraie_attaque.sum()
        resultats.append((tau_conf, tau_err_pct, n_signalees, precision, rappel))
        print(f"tau_conf={tau_conf} | seuil_err=percentile_{tau_err_pct} | signalees={n_signalees} | precision={precision:.3f} | rappel={rappel:.3f}")

df_balayage = pd.DataFrame(resultats, columns=['tau_conf','percentile_erreur','n_signalees','precision','rappel'])
df_balayage.to_csv(f'{checkpoint_dir_v4}/balayage_seuils_pipeline_v4.csv', index=False)
print("\nSauvegarde : balayage_seuils_pipeline_v4.csv")
print("\nMeilleur compromis precision/rappel :")
df_balayage['score'] = df_balayage['precision'] + df_balayage['rappel']
print(df_balayage.sort_values('score', ascending=False).head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Balayage reel de plusieurs couples de seuils (confiance, erreur) ===


ValueError: operands could not be broadcast together with shapes (30922,) (15589,) 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.simplefilter('ignore')
import pickle, joblib
import numpy as np
import pandas as pd
from tensorflow.keras import models as keras_models
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
window_size = 20

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
feature_cols = saved['feature_cols']

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')

ATTAQUE_RETIREE = 'Evil_Twin'  # jamais montree a l'entrainement

def build_windows(df, exclude_attack=None, only_attack=None):
    X_max, y_binaire = [], []
    for file_id, group in df.groupby('source_file', sort=False):
        attack = group['source_attack'].iloc[0]
        if exclude_attack and attack == exclude_attack:
            continue
        if only_attack and attack != only_attack:
            continue
        feats = group[feature_cols].values.astype(np.float32)
        labels = group['Label'].values
        n_win = len(group) // window_size
        for i in range(n_win):
            s = i * window_size
            window = feats[s:s+window_size]
            wl = labels[s:s+window_size]
            X_max.append(window.max(axis=0))
            y_binaire.append(1 if (wl != 'Normal').any() else 0)
    return np.array(X_max), np.array(y_binaire)

print(f"Entrainement SANS jamais montrer l'attaque : {ATTAQUE_RETIREE}")
X_train, y_train = build_windows(df_encoded, exclude_attack=ATTAQUE_RETIREE)
X_train_tr, X_val, y_train_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_tr, y_train_tr)

ae_binaire = XGBClassifier(n_estimators=300, max_depth=6, random_state=42, eval_metric='logloss')
ae_binaire.fit(X_train_sm, y_train_sm)

print(f"\nTest EXCLUSIVEMENT sur l'attaque jamais vue : {ATTAQUE_RETIREE}")
X_inedit, y_inedit_vrai = build_windows(df_encoded, only_attack=ATTAQUE_RETIREE)
proba_inedit = ae_binaire.predict_proba(X_inedit)[:, 1]
taux_detection_inedit = (proba_inedit > 0.5).mean()

X_normal_test, y_normal_test = build_windows(df_encoded, only_attack=None)
mask_normal = y_normal_test == 0

print(f"\n=== Resultat du test zero-day sur {ATTAQUE_RETIREE} ===")
print(f"Nombre de fenetres de l'attaque inedite testees : {len(X_inedit)}")
print(f"Taux de detection de l'attaque jamais vue a l'entrainement : {taux_detection_inedit:.4f}")
print(f"Score de confiance moyen attribue : {proba_inedit.mean():.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entrainement SANS jamais montrer l'attaque : Evil_Twin

Test EXCLUSIVEMENT sur l'attaque jamais vue : Evil_Twin

=== Resultat du test zero-day sur Evil_Twin ===
Nombre de fenetres de l'attaque inedite testees : 12307
Taux de detection de l'attaque jamais vue a l'entrainement : 0.0488
Score de confiance moyen attribue : 0.1048


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.simplefilter('ignore')
import pickle, numpy as np, pandas as pd
from tensorflow.keras import models as keras_models

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

data_hybrid = np.load(f'{checkpoint_dir_v4}/hybrid_proba_v4.npz')
hybrid_proba = 0.4*data_hybrid['xgb_proba'] + 0.4*data_hybrid['lstm_proba'] + 0.2*data_hybrid['gnn_proba']
S_sup = hybrid_proba.max(axis=1)
y_test = data_hybrid['y_test']

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
le_max = saved['le']; feature_cols = saved['feature_cols']

with open(f'{checkpoint_dir_v4}/scaler_autoencoder_v4.pkl', 'rb') as f:
    ae_saved = pickle.load(f)
scaler_ae = ae_saved['scaler']
ae_model = keras_models.load_model(f'{checkpoint_dir_v4}/autoencoder_v4.keras')

# Reconstruire exactement la meme sequence de test que celle utilisee pour l'Hybride,
# pour que err_test et S_sup correspondent fenetre a fenetre
df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')
window_size = 20
X_seq_list, labels_list = [], []
for file_id, group in df_encoded.groupby('source_file', sort=False):
    feats = group[feature_cols].values.astype(np.float32)
    lab = group['Label'].values
    attack = group['source_attack'].iloc[0]
    n_win = len(group) // window_size
    for i in range(n_win):
        s = i * window_size
        X_seq_list.append(feats[s:s+window_size])
        wl = lab[s:s+window_size]
        labels_list.append(attack if (wl != 'Normal').any() else 'Normal')

from sklearn.model_selection import train_test_split
X_seq_all = np.array(X_seq_list)
y_all_encoded = le_max.transform(labels_list)
_, X_seq_test_new, _, y_test_new = train_test_split(
    X_seq_all, y_all_encoded, test_size=0.2, random_state=42, stratify=y_all_encoded)

print("Verification longueur :", len(X_seq_test_new), "==", len(S_sup))

n_t, n_f = X_seq_test_new.shape[1], X_seq_test_new.shape[2]
X_seq_scaled = scaler_ae.transform(X_seq_test_new.reshape(-1, n_f)).reshape(X_seq_test_new.shape)
X_reconstruit = ae_model.predict(X_seq_scaled, batch_size=256)
err_test = np.mean((X_seq_scaled - X_reconstruit)**2, axis=(1,2))

label_normal_idx = le_max.transform(['Normal'])[0]
vraie_attaque = (y_test_new != label_normal_idx)

print("\n=== Balayage reel de plusieurs couples de seuils (confiance, erreur) ===")
resultats = []
for tau_conf in [0.3, 0.4, 0.5, 0.6]:
    for tau_err_pct in [75, 85, 90, 95]:
        tau_err = np.percentile(err_test, tau_err_pct)
        signal = (S_sup < tau_conf) & (err_test > tau_err)
        n_signalees = signal.sum()
        if n_signalees == 0:
            continue
        precision = (signal & vraie_attaque).sum() / n_signalees
        rappel = (signal & vraie_attaque).sum() / vraie_attaque.sum()
        resultats.append((tau_conf, tau_err_pct, n_signalees, precision, rappel))
        print(f"tau_conf={tau_conf} | seuil_err=percentile_{tau_err_pct} | signalees={n_signalees} | precision={precision:.3f} | rappel={rappel:.3f}")

df_balayage = pd.DataFrame(resultats, columns=['tau_conf','percentile_erreur','n_signalees','precision','rappel'])
df_balayage.to_csv(f'{checkpoint_dir_v4}/balayage_seuils_pipeline_v4.csv', index=False)
df_balayage['score'] = df_balayage['precision'] + df_balayage['rappel']
print("\nMeilleur compromis :")
print(df_balayage.sort_values('score', ascending=False).head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verification longueur : 30922 == 30922
121/121 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

=== Balayage reel de plusieurs couples de seuils (confiance, erreur) ===
tau_conf=0.3 | seuil_err=percentile_75 | signalees=11 | precision=0.636 | rappel=0.000
tau_conf=0.3 | seuil_err=percentile_85 | signalees=9 | precision=0.556 | rappel=0.000
tau_conf=0.3 | seuil_err=percentile_90 | signalees=7 | precision=0.429 | rappel=0.000
tau_conf=0.3 | seuil_err=percentile_95 | signalees=6 | precision=0.333 | rappel=0.000
tau_conf=0.4 | seuil_err=percentile_75 | signalees=150 | precision=0.527 | rappel=0.005
tau_conf=0.4 | seuil_err=percentile_85 | signalees=106 | precision=0.547 | rappel=0.004
tau_conf=0.4 | seuil_err=percentile_90 | signalees=81 | precision=0.556 | rappel=0.003
tau_conf=0.4 | seuil_err=percentile_95 | signalees=47 | precision=0.596 | rappel=0.002
tau_conf=0.5 | seuil_e

In [ ]:
import pandas as pd
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
cm_df = pd.read_csv(f'{checkpoint_dir_v4}/matrice_confusion_fichiers_v4.csv', index_col=0)
print(cm_df.to_string())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4/matrice_confusion_fichiers_v4.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
for f in sorted(os.listdir(checkpoint_dir_v4)):
    print(f)

Mounted at /content/drive
agents_decisions_correctives_v4.pkl
autoencoder_auc_par_attaque_v4.csv
autoencoder_v4.keras
balayage_frein_normal_v4.csv
balayage_seuils_pipeline_v4.csv
base_connaissances_sortie_decisions_v4.parquet
base_connaissances_sortie_reentrainement_v4.csv
base_connaissances_v4.parquet
comparaison_poids_hybride_v4.csv
confusion_matrix_gnn_v4.csv
confusion_matrix_hybrid_v4.csv
confusion_matrix_lstm_v4.csv
confusion_matrix_xgb_combo_v4.csv
cv_fichiers_pli0_v4.npz
cv_fichiers_pli1_v4.npz
cv_fichiers_pli2_v4.npz
cv_fichiers_pli3_v4.npz
cv_fichiers_pli4_v4.npz
cv_fichiers_resultats_v4.csv
decisions_correctives_v4.parquet
decomposition_hybride_ablation_v4.csv
dependance_colonnes_globale_v4.csv
df_encoded_v4.parquet
df_raw_v4.parquet
diagnostic_rca_topk_mrr_v4.csv
env_auto_guerison_etat_v4.parquet
env_auto_guerison_v4.parquet
fichiers_test_auto_guerison_v4.csv
fichiers_train_auto_guerison_v4.csv
gnn_sum_v4.pt
gnn_v4.pt
hybrid_proba_v4.npz
le_combo_v4.pkl
le_gnn_v4.pkl
lstm_v4

In [ ]:
import pandas as pd
checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'
cm_df = pd.read_csv(f'{checkpoint_dir_v4}/matrice_confusion_fichiers_v4.csv', index_col=0)
print(cm_df.to_string())


                  Botnet  Deauth  Disas  Evil_Twin  Kr00k  Krack  Malware  Normal  Re-Assoc  Rogue_AP  SQL_Injection  SSDP  SSH  Website_spoofing
Botnet               554       0      1          5      0      0       62     413         1         0              0     0    0                 0
Deauth                 1     620     15          0     35      0        0     184        28         0              0     0    1                 1
Disas                  2     377   1059          0    266      1        0     185        64         0              9     0    1                 2
Evil_Twin              2       0      0        703      0      0        0     347         0         0              1     0    1                 0
Kr00k                  0      33   1018          0   1275      1        0     117         5         2              2     0    1                 0
Krack                  0       0      0          3      2    533        0     141         1         0              0     0  

In [ ]:
import warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense

window_size = 20
ATTAQUE_RETIREE = 'Evil_Twin'

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')

def build_windows_excl(df, exclude_attack=None, only_attack=None):
    X_max, X_seq, y_bin = [], [], []
    for file_id, group in df.groupby('source_file', sort=False):
        attack = group['source_attack'].iloc[0]
        if exclude_attack and attack == exclude_attack: continue
        if only_attack and attack != only_attack: continue
        feats = group[feature_cols].values.astype(np.float32)
        labels = group['Label'].values
        n_win = len(group) // window_size
        for i in range(n_win):
            s = i * window_size
            window = feats[s:s+window_size]
            wl = labels[s:s+window_size]
            X_max.append(window.max(axis=0))
            X_seq.append(window)
            y_bin.append(1 if (wl != 'Normal').any() else 0)
    return np.array(X_max), np.array(X_seq), np.array(y_bin)

print("Construction des fenetres, Evil_Twin exclu de l'entrainement...")
X_max_tr, X_seq_tr, y_bin_tr = build_windows_excl(df_encoded, exclude_attack=ATTAQUE_RETIREE)
X_max_tr2, X_max_val, X_seq_tr2, X_seq_val, y_tr2, y_val = train_test_split(
    X_max_tr, X_seq_tr, y_bin_tr, test_size=0.2, random_state=42, stratify=y_bin_tr)

# Volet supervise : XGBoost binaire, sans Evil_Twin
sm = SMOTE(random_state=42)
X_max_tr2_sm, y_tr2_sm = sm.fit_resample(X_max_tr2, y_tr2)
xgb_bin = XGBClassifier(n_estimators=300, max_depth=6, random_state=42, eval_metric='logloss')
xgb_bin.fit(X_max_tr2_sm, y_tr2_sm)

# Volet non supervise : autoencodeur, entraine uniquement sur le trafic normal, sans Evil_Twin
n_t, n_f = X_seq_tr2.shape[1], X_seq_tr2.shape[2]
scaler = StandardScaler()
X_seq_tr2_scaled = scaler.fit_transform(X_seq_tr2.reshape(-1, n_f)).reshape(X_seq_tr2.shape)
X_seq_val_scaled = scaler.transform(X_seq_val.reshape(-1, n_f)).reshape(X_seq_val.shape)

mask_normal = (y_tr2 == 0)
X_seq_normal = X_seq_tr2_scaled[mask_normal]

inp = Input(shape=(n_t, n_f))
enc = LSTM(32)(inp)
dec = RepeatVector(n_t)(enc)
dec = LSTM(32, return_sequences=True)(dec)
out = TimeDistributed(Dense(n_f))(dec)
ae_model = Model(inp, out)
ae_model.compile(optimizer='adam', loss='mse')
print("Entrainement de l'autoencodeur, sans Evil_Twin...")
ae_model.fit(X_seq_normal, X_seq_normal, epochs=20, batch_size=256, validation_split=0.1, verbose=0)

# Seuils, calibres sur la validation (qui ne contient pas non plus Evil_Twin)
proba_val = xgb_bin.predict_proba(X_max_val)[:, 1]
X_recon_val = ae_model.predict(X_seq_val_scaled, batch_size=256, verbose=0)
err_val = np.mean((X_seq_val_scaled - X_recon_val)**2, axis=(1,2))
tau_conf = 0.5
tau_err = np.percentile(err_val, 90)
print(f"Seuils retenus : tau_conf={tau_conf} | tau_err (90e percentile)={tau_err:.4f}")

# Test exclusivement sur Evil_Twin, jamais vu ni par le XGBoost ni par l'autoencodeur
X_max_inedit, X_seq_inedit, _ = build_windows_excl(df_encoded, only_attack=ATTAQUE_RETIREE)
X_seq_inedit_scaled = scaler.transform(X_seq_inedit.reshape(-1, n_f)).reshape(X_seq_inedit.shape)

proba_inedit = xgb_bin.predict_proba(X_max_inedit)[:, 1]
X_recon_inedit = ae_model.predict(X_seq_inedit_scaled, batch_size=256, verbose=0)
err_inedit = np.mean((X_seq_inedit_scaled - X_recon_inedit)**2, axis=(1,2))

confiance_faible = proba_inedit < tau_conf
erreur_elevee = err_inedit > tau_err
signale_pipeline = confiance_faible | erreur_elevee

print(f"\nFenetres de {ATTAQUE_RETIREE} testees : {len(X_max_inedit)}")
print(f"Detectees par le XGBoost seul (confiance faible)   : {confiance_faible.mean()*100:.2f} %")
print(f"Detectees par l'autoencodeur seul (erreur elevee)   : {erreur_elevee.mean()*100:.2f} %")
print(f"Detectees par le pipeline complet (XGBoost OU AE)   : {signale_pipeline.mean()*100:.2f} %")

Construction des fenetres, Evil_Twin exclu de l'entrainement...
Entrainement de l'autoencodeur, sans Evil_Twin...
Seuils retenus : tau_conf=0.5 | tau_err (90e percentile)=0.9288

Fenetres de Evil_Twin testees : 12307
Detectees par le XGBoost seul (confiance faible)   : 95.12 %
Detectees par l'autoencodeur seul (erreur elevee)   : 7.47 %
Detectees par le pipeline complet (XGBoost OU AE)   : 95.99 %


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pickle

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
feature_cols = saved['feature_cols']

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pickle

checkpoint_dir_v4 = '/content/drive/MyDrive/EXTRAITFULL/SIMULATION_V4'

with open(f'{checkpoint_dir_v4}/le_combo_v4.pkl', 'rb') as f:
    saved = pickle.load(f)
feature_cols = saved['feature_cols']

import warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense

window_size = 20
ATTAQUE_RETIREE = 'Evil_Twin'

df_encoded = pd.read_parquet(f'{checkpoint_dir_v4}/df_encoded_v4.parquet')

def build_windows_excl(df, exclude_attack=None, only_attack=None):
    X_max, X_seq, y_bin = [], [], []
    for file_id, group in df.groupby('source_file', sort=False):
        attack = group['source_attack'].iloc[0]
        if exclude_attack and attack == exclude_attack: continue
        if only_attack and attack != only_attack: continue
        feats = group[feature_cols].values.astype(np.float32)
        labels = group['Label'].values
        n_win = len(group) // window_size
        for i in range(n_win):
            s = i * window_size
            window = feats[s:s+window_size]
            wl = labels[s:s+window_size]
            X_max.append(window.max(axis=0))
            X_seq.append(window)
            y_bin.append(1 if (wl != 'Normal').any() else 0)
    return np.array(X_max), np.array(X_seq), np.array(y_bin)

print("Construction des fenetres, Evil_Twin exclu de l'entrainement...")
X_max_tr, X_seq_tr, y_bin_tr = build_windows_excl(df_encoded, exclude_attack=ATTAQUE_RETIREE)
X_max_tr2, X_max_val, X_seq_tr2, X_seq_val, y_tr2, y_val = train_test_split(
    X_max_tr, X_seq_tr, y_bin_tr, test_size=0.2, random_state=42, stratify=y_bin_tr)

# Volet supervise : XGBoost binaire, sans Evil_Twin
sm = SMOTE(random_state=42)
X_max_tr2_sm, y_tr2_sm = sm.fit_resample(X_max_tr2, y_tr2)
xgb_bin = XGBClassifier(n_estimators=300, max_depth=6, random_state=42, eval_metric='logloss')
xgb_bin.fit(X_max_tr2_sm, y_tr2_sm)

# Volet non supervise : autoencodeur, entraine uniquement sur le trafic normal, sans Evil_Twin
n_t, n_f = X_seq_tr2.shape[1], X_seq_tr2.shape[2]
scaler = StandardScaler()
X_seq_tr2_scaled = scaler.fit_transform(X_seq_tr2.reshape(-1, n_f)).reshape(X_seq_tr2.shape)
X_seq_val_scaled = scaler.transform(X_seq_val.reshape(-1, n_f)).reshape(X_seq_val.shape)

mask_normal = (y_tr2 == 0)
X_seq_normal = X_seq_tr2_scaled[mask_normal]

inp = Input(shape=(n_t, n_f))
enc = LSTM(32)(inp)
dec = RepeatVector(n_t)(enc)
dec = LSTM(32, return_sequences=True)(dec)
out = TimeDistributed(Dense(n_f))(dec)
ae_model = Model(inp, out)
ae_model.compile(optimizer='adam', loss='mse')
print("Entrainement de l'autoencodeur, sans Evil_Twin...")
ae_model.fit(X_seq_normal, X_seq_normal, epochs=20, batch_size=256, validation_split=0.1, verbose=0)

# Seuils, calibres sur la validation (qui ne contient pas non plus Evil_Twin)
X_recon_val = ae_model.predict(X_seq_val_scaled, batch_size=256, verbose=0)
err_val = np.mean((X_seq_val_scaled - X_recon_val)**2, axis=(1,2))
tau_conf = 0.5
tau_err = np.percentile(err_val, 90)
print(f"Seuils retenus : tau_conf={tau_conf} | tau_err (90e percentile)={tau_err:.4f}")

# Test exclusivement sur Evil_Twin, jamais vu ni par le XGBoost ni par l'autoencodeur
X_max_inedit, X_seq_inedit, _ = build_windows_excl(df_encoded, only_attack=ATTAQUE_RETIREE)
X_seq_inedit_scaled = scaler.transform(X_seq_inedit.reshape(-1, n_f)).reshape(X_seq_inedit.shape)

proba_inedit = xgb_bin.predict_proba(X_max_inedit)[:, 1]
X_recon_inedit = ae_model.predict(X_seq_inedit_scaled, batch_size=256, verbose=0)
err_inedit = np.mean((X_seq_inedit_scaled - X_recon_inedit)**2, axis=(1,2))

# Une fenetre est signalee si le XGBoost est confiant qu'il s'agit d'une attaque,
# OU si l'autoencodeur la reconstruit mal
attaque_detectee_xgb = proba_inedit > tau_conf
erreur_elevee = err_inedit > tau_err
signale_pipeline = attaque_detectee_xgb | erreur_elevee

print(f"\nFenetres de {ATTAQUE_RETIREE} testees : {len(X_max_inedit)}")
print(f"Detectees par le XGBoost seul (confiance elevee)    : {attaque_detectee_xgb.mean()*100:.2f} %")
print(f"Detectees par l'autoencodeur seul (erreur elevee)   : {erreur_elevee.mean()*100:.2f} %")
print(f"Detectees par le pipeline complet (XGBoost OU AE)   : {signale_pipeline.mean()*100:.2f} %")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Construction des fenetres, Evil_Twin exclu de l'entrainement...
Entrainement de l'autoencodeur, sans Evil_Twin...
Seuils retenus : tau_conf=0.5 | tau_err (90e percentile)=0.9486

Fenetres de Evil_Twin testees : 12307
Detectees par le XGBoost seul (confiance elevee)    : 4.88 %
Detectees par l'autoencodeur seul (erreur elevee)   : 7.25 %
Detectees par le pipeline complet (XGBoost OU AE)   : 11.30 %
